# MAGIC-I: TPMT/NUDT15 pharmacogenomics and MRD analysis

Analysis of the MAGIC-I paediatric leukaemia cohort, examining TPMT and NUDT15 expression in relation to lineage, MRD status, pathway activity, and germline genotype, with a partial replication comparison against the TARGET cohort. Companion analysis to the TARGET pipeline; thesis MRD/TPMT/NUDT15 focus is a related but separate workstream.

Raw sample-level identifiers, genotype calls, and clinical tables are not displayed inline — only de-identified summaries and plots. Full per-patient results are written out to CSV files listed at the point each is generated. Input data files are expected under a local data/ folder and are not included in this repository.

#### Load manifest and filter for leukaemia ####

In [ ]:
import pandas as pd
import os

tracking_path = "data/MAGIC-I_Master-File.csv"

tracking = pd.read_csv(tracking_path)  
print(tracking.shape)
print(tracking.columns.tolist()[:10])
print(tracking['Study ID'].head(10))

In [ ]:
manifest_path = "data/cohort-counts-manifest.csv"
manifest = pd.read_csv(manifest_path)

print(manifest.shape)
print(manifest.columns.tolist())
print(manifest['disease'].unique())

merged = manifest.merge(
    tracking,
    left_on='study_id',
    right_on='Study ID',
    how='left'
)
print(merged.shape)
print(f"Matched: {merged['Study ID'].notna().sum()} of {len(manifest)}")

In [ ]:
# Clean whitespace in D/R column first
merged['D/R'] = merged['D/R'].str.strip()

# Restrict to diagnosis-only timepoint (excludes relapse samples)
manifest_diag = merged[merged['D/R'] == 'Diag'].copy()
print(f"Diagnosis-only: n={len(manifest_diag)} of {len(merged)}")

# Narrow definition: lymphoid leukaemias/lymphomas (ALL + LBL + MPAL) — excludes AML
leukaemia_keywords = ['ALL', 'MPAL', 'LBL']
mask_narrow = (
    manifest_diag['disease'].str.contains('|'.join(leukaemia_keywords), case=False, na=False)
    & ~manifest_diag['disease'].str.contains('AML', case=False, na=False)
)
leukaemia_manifest_narrow = manifest_diag[mask_narrow]

# Broad definition: narrow + AML
mask_broad = (
    manifest_diag['disease'].str.contains('|'.join(leukaemia_keywords + ['AML']), case=False, na=False)
)
leukaemia_manifest_broad = manifest_diag[mask_broad]

print("\nNARROW (ALL + LBL + MPAL, no AML, diagnosis-only):")
print(leukaemia_manifest_narrow[['sample', 'disease']])
print(f"{len(leukaemia_manifest_narrow)} of {len(manifest_diag)} samples\n")

print("BROAD (narrow + AML, diagnosis-only):")
print(leukaemia_manifest_broad[['sample', 'disease']])
print(f"{len(leukaemia_manifest_broad)} of {len(manifest_diag)} samples")

In [ ]:
counts_path = "data/cohort-counts.tsv"
counts = pd.read_csv(counts_path, sep="\t", index_col=0)

print(counts.shape)
print(counts.columns[:10].tolist())
print(counts.index[:10].tolist())

In [ ]:
# Narrow leukaemia-only expression matrix
counts_leukaemia_narrow = counts[leukaemia_manifest_narrow['sample']]

# Broad (leukaemia + LBL) expression matrix
counts_leukaemia_broad = counts[leukaemia_manifest_broad['sample']]

print("Narrow:", counts_leukaemia_narrow.shape)
print("Broad:", counts_leukaemia_broad.shape)

# quick sanity check - same gene index as original, only columns changed
print(counts_leukaemia_narrow.index.equals(counts.index))

In [ ]:
out_dir = "data/"

leukaemia_manifest_narrow.to_csv(out_dir + "leukaemia-manifest-narrow.csv", index=False)
leukaemia_manifest_broad.to_csv(out_dir + "leukaemia-manifest-broad.csv", index=False)
counts_leukaemia_narrow.to_csv(out_dir + "leukaemia-counts-narrow.tsv", sep="\t")
counts_leukaemia_broad.to_csv(out_dir + "leukaemia-counts-broad.tsv", sep="\t")

#### Cancer type and clinical outcome comparison ####

In [ ]:
# TPMT and NUDT15 Ensembl gene IDs
gene_ids = {
    'TPMT': 'ENSG00000137364',
    'NUDT15': 'ENSG00000136159'
}

tpmt_nudt15 = counts_leukaemia_narrow.loc[list(gene_ids.values())].T
tpmt_nudt15.columns = list(gene_ids.keys())
tpmt_nudt15 = tpmt_nudt15.reset_index().rename(columns={'index': 'sample'})

# merge with disease/subtype labels
tpmt_nudt15 = tpmt_nudt15.merge(leukaemia_manifest_narrow[['sample', 'disease']], on='sample')

print(tpmt_nudt15)

In [ ]:
# collapse disease labels to broad lineage groups
def broad_lineage(d):
    d = d.upper()
    if 'MPAL' in d:
        return 'MPAL'
    if 'AML' in d:
        return 'AML'
    if 'T-ALL' in d or 'T-LBL' in d:
        return 'T-ALL'
    if 'B-ALL' in d or 'B, ALL' in d or 'DS-B-ALL' in d or 'B-LBL' in d:
        return 'B-ALL'
    return 'Other'

tpmt_nudt15['lineage'] = tpmt_nudt15['disease'].apply(broad_lineage)
print(tpmt_nudt15['lineage'].value_counts())

In [ ]:
nan_counts = counts_leukaemia_narrow.isna().sum()
print(nan_counts[nan_counts > 0])

nan_genes = counts_leukaemia_narrow.isna().any(axis=1).sum()
print(f"\n{nan_genes} genes have at least one NaN out of {counts_leukaemia_narrow.shape[0]} total")

In [ ]:
counts_clean = counts_leukaemia_narrow.dropna()
print(counts_clean.shape)
print(counts_clean.isna().sum().sum())  # should be 0

# now proceed to DESeq2
counts_for_deseq = counts_clean.T
counts_for_deseq = counts_for_deseq.astype(int)

metadata = tpmt_nudt15[['sample', 'lineage']].set_index('sample')
metadata = metadata.loc[counts_for_deseq.index]

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference

inference = DefaultInference(n_cpus=4)
dds = DeseqDataSet(
    counts=counts_for_deseq,
    metadata=metadata,
    design_factors='lineage',
    inference=inference
)

dds.deseq2()
dds.vst()
vst_counts = dds.layers['vst_counts']
vst_df = pd.DataFrame(vst_counts, index=counts_for_deseq.index, columns=counts_for_deseq.columns)

print(vst_df.shape)

In [ ]:
tpmt_nudt15_vst = vst_df[['ENSG00000137364', 'ENSG00000136159']].copy()
tpmt_nudt15_vst.columns = ['TPMT_vst', 'NUDT15_vst']
tpmt_nudt15_vst = tpmt_nudt15_vst.reset_index().rename(columns={'index': 'sample'})
tpmt_nudt15_vst = tpmt_nudt15_vst.merge(tpmt_nudt15[['sample', 'lineage']], on='sample')

print(tpmt_nudt15_vst.sort_values('lineage'))

In [ ]:
from scipy import stats

# descriptive summary for all groups
summary = tpmt_nudt15_vst.groupby('lineage')[['TPMT_vst', 'NUDT15_vst']].agg(['mean', 'std', 'count'])
print(summary)

# formal test: B-ALL vs T-ALL only (adequate n)
ball = tpmt_nudt15_vst[tpmt_nudt15_vst['lineage'] == 'B-ALL']
tall = tpmt_nudt15_vst[tpmt_nudt15_vst['lineage'] == 'T-ALL']

tpmt_stat, tpmt_p = stats.mannwhitneyu(ball['TPMT_vst'], tall['TPMT_vst'])
nudt15_stat, nudt15_p = stats.mannwhitneyu(ball['NUDT15_vst'], tall['NUDT15_vst'])

print(f"\nTPMT B-ALL vs T-ALL: p={tpmt_p:.4f}")
print(f"NUDT15 B-ALL vs T-ALL: p={nudt15_p:.4f}")

**TPMT** - MPAL has the highest mean (11.39, n=1, single sample so no meaningful spread), followed by B-ALL (9.25) and T-ALL(9.15), which overlap substantially given their spread (std 0.58 and 1.03 respectively). The B-ALL vs T-ALL difference is not significant (p=0.7279).

**NUDT15**- all three groups sit in narrow band (~9.45-9.48), no meaningful separation by lineage at all. Tracks with the TARGET result that NUDT15 shows no significant MRD association either. NUDT15 may just be more lineage/outcome invariant in general.


In [ ]:
print(f"manifest rows: {len(manifest)}, unique samples: {manifest['sample'].nunique()}, unique study IDs: {manifest['study_id'].nunique()}")


In [ ]:
print(leukaemia_manifest_narrow['Fusion'].value_counts(dropna=False))

In [ ]:
print(leukaemia_manifest_narrow['Fusion'].unique())

# check if NaN correlates with FISH/array/NGS not being done at all
print(leukaemia_manifest_narrow[leukaemia_manifest_narrow['Fusion'].isna()][['FISH SOC', 'array SOC', 'RNA NGS SOC', 'DNA NGS SOC']].apply(pd.Series.value_counts))

In [ ]:
fusion_clean = leukaemia_manifest_narrow['Fusion'].str.strip()
fusion_clean = fusion_clean.replace({'No': 'Fusion-negative', 'no fusion': 'Fusion-negative'})

leukaemia_manifest_narrow['fusion_clean'] = fusion_clean
print(leukaemia_manifest_narrow['fusion_clean'].value_counts(dropna=False))

In [ ]:
subtype_summary = leukaemia_manifest_narrow[['sample', 'fusion_clean']].merge(
    tpmt_nudt15_vst[['sample', 'TPMT_vst', 'NUDT15_vst']], on='sample'
)

# groups with n>=2
counts = subtype_summary['fusion_clean'].value_counts()
adequate_groups = counts[counts >= 2].index

print("Groups with n>=2 samples:")
print(subtype_summary[subtype_summary['fusion_clean'].isin(adequate_groups)]
      .groupby('fusion_clean')[['TPMT_vst', 'NUDT15_vst']].agg(['mean', 'std', 'count']))

print("\nAll samples (for appendix):")
print(subtype_summary.sort_values('fusion_clean'))

In [ ]:
from scipy.stats import pearsonr, spearmanr

# TPMT-NUDT15 correlation across all 46 samples
r, p = spearmanr(tpmt_nudt15_vst['TPMT_vst'], tpmt_nudt15_vst['NUDT15_vst'])
print(f"TPMT-NUDT15 Spearman correlation: rho={r:.3f}, p={p:.4f}")

print(leukaemia_manifest_narrow[['sample', 'D/R', 'age', 'Sex']].head(20))

print("\nD/R value counts:")
print(leukaemia_manifest_narrow['D/R'].value_counts(dropna=False))

print("\nAge summary:")
print(leukaemia_manifest_narrow['age'].describe())

TPMT-NUDT15 correlation: essentially 0 (rho = 0.021, p=0.8924). The two genes don't co-vary at all in this cohort. Confirmst they're regulating independently, consistent of them being separate genes in the thiopurine pathway. 


In [ ]:
# age correlation with TPMT/NUDT15
age_analysis = leukaemia_manifest_narrow[['sample', 'age']].merge(tpmt_nudt15_vst[['sample','TPMT_vst','NUDT15_vst']], on='sample')

r_tpmt, p_tpmt = spearmanr(age_analysis['age'], age_analysis['TPMT_vst'])
r_nudt, p_nudt = spearmanr(age_analysis['age'], age_analysis['NUDT15_vst'])
print(f"\nAge vs TPMT: rho={r_tpmt:.3f}, p={p_tpmt:.4f}")
print(f"Age vs NUDT15: rho={r_nudt:.3f}, p={p_nudt:.4f}")

In [ ]:
# library size = total counts per sample
lib_sizes = counts_leukaemia_narrow.sum(axis=0)
print(lib_sizes.describe())

# correlate raw TPMT/NUDT15 counts with library size (before VST correction)
raw_tpmt = counts_leukaemia_narrow.loc['ENSG00000137364']
raw_nudt15 = counts_leukaemia_narrow.loc['ENSG00000136159']

r_lib_tpmt, p_lib_tpmt = spearmanr(lib_sizes, raw_tpmt)
r_lib_nudt15, p_lib_nudt15 = spearmanr(lib_sizes, raw_nudt15)
print(f"\nLibrary size vs raw TPMT: rho={r_lib_tpmt:.3f}, p={p_lib_tpmt:.4f}")
print(f"Library size vs raw NUDT15: rho={r_lib_nudt15:.3f}, p={p_lib_nudt15:.4f}")

In [ ]:
r_lib_tpmt_vst, p_lib_tpmt_vst = spearmanr(lib_sizes, vst_df['ENSG00000137364'])
r_lib_nudt15_vst, p_lib_nudt15_vst = spearmanr(lib_sizes, vst_df['ENSG00000136159'])
print(f"Library size vs VST TPMT: rho={r_lib_tpmt_vst:.3f}, p={p_lib_tpmt_vst:.4f}")
print(f"Library size vs VST NUDT15: rho={r_lib_nudt15_vst:.3f}, p={p_lib_nudt15_vst:.4f}")

In [ ]:
print(lib_sizes.sort_values().head(3))

In [ ]:
# Derive QC exclusions from  QC conclusion values
print(leukaemia_manifest_narrow[['sample', 'QC conclusion']].drop_duplicates())

# Samples with an explicit fail flag
qc_fail_samples = leukaemia_manifest_narrow[
    leukaemia_manifest_narrow['QC conclusion'].str.contains('FAIL', case=False, na=False)
]['sample'].tolist()

# Samples with an ambiguous/borderline flag — review separately, don't auto-exclude
qc_borderline_samples = leukaemia_manifest_narrow[
    leukaemia_manifest_narrow['QC conclusion'].isin(['No great'])
]['sample'].tolist()

print(f"\nQC fail samples (derived): {qc_fail_samples}")
print(f"QC borderline samples (derived): {qc_borderline_samples}")

In [ ]:
def broad_lineage(d):
    d = d.upper()
    if 'MPAL' in d: return 'MPAL'
    if 'AML' in d: return 'AML'
    if 'T-ALL' in d or 'T-LBL' in d: return 'T-ALL'
    if 'B-ALL' in d or 'B, ALL' in d or 'DS-B-ALL' in d or 'B-LBL' in d: return 'B-ALL'
    return 'Other'

leukaemia_manifest_narrow['lineage'] = leukaemia_manifest_narrow['disease'].apply(broad_lineage)
print(leukaemia_manifest_narrow['lineage'].value_counts())

qc_exclude_strict = ['A3873043']  # confirmed QC failure ("FAILED?")

tpmt_nudt15_vst_strict = tpmt_nudt15_vst[~tpmt_nudt15_vst['sample'].isin(qc_exclude_strict)]

In [ ]:
tpmt_nudt15_vst_strict = tpmt_nudt15_vst[~tpmt_nudt15_vst['sample'].isin(qc_exclude_strict)]

# 1. lineage comparison
summary_strict = tpmt_nudt15_vst_strict.groupby('lineage')[['TPMT_vst', 'NUDT15_vst']].agg(['mean','std','count'])
print("Lineage summary (strict QC):")
print(summary_strict)

ball = tpmt_nudt15_vst_strict[tpmt_nudt15_vst_strict['lineage']=='B-ALL']
tall = tpmt_nudt15_vst_strict[tpmt_nudt15_vst_strict['lineage']=='T-ALL']
tpmt_stat, tpmt_p = stats.mannwhitneyu(ball['TPMT_vst'], tall['TPMT_vst'])
nudt15_stat, nudt15_p = stats.mannwhitneyu(ball['NUDT15_vst'], tall['NUDT15_vst'])
print(f"\nTPMT B-ALL vs T-ALL (strict): p={tpmt_p:.4f}")
print(f"NUDT15 B-ALL vs T-ALL (strict): p={nudt15_p:.4f}")

# 2. TPMT-NUDT15 correlation
r, p = spearmanr(tpmt_nudt15_vst_strict['TPMT_vst'], tpmt_nudt15_vst_strict['NUDT15_vst'])
print(f"\nTPMT-NUDT15 correlation (strict): rho={r:.3f}, p={p:.4f}")

# 3. age correlation
age_strict = leukaemia_manifest_narrow[~leukaemia_manifest_narrow['sample'].isin(qc_exclude_strict)][['sample','age']].merge(
    tpmt_nudt15_vst_strict[['sample','TPMT_vst','NUDT15_vst']], on='sample'
)
r_a, p_a = spearmanr(age_strict['age'], age_strict['TPMT_vst'])
r_n, p_n = spearmanr(age_strict['age'], age_strict['NUDT15_vst'])
print(f"\nAge vs TPMT (strict): rho={r_a:.3f}, p={p_a:.4f}")
print(f"Age vs NUDT15 (strict): rho={r_n:.3f}, p={p_n:.4f}")

In [ ]:
check_cols = ['Infiltration', 'Sex', 'Disease Category', 'SNOMED CT', 'PMC /RNAseq', 'PMC WES']
for col in check_cols:
    print(f"=== {col} ===")
    print(leukaemia_manifest_narrow[col].value_counts(dropna=False).head(10))
    print()

In [ ]:
print(f"PMC RNAseq flagged: {(leukaemia_manifest_narrow['PMC /RNAseq'] == 'Y').sum()} samples")
print(f"PMC WES flagged: {(leukaemia_manifest_narrow['PMC WES'] == 'Y (necker)').sum()} samples")


#### ALLSorts and Polonen Subtyping ####

In [ ]:
# Build ALLSorts input from the corrected B-ALL subset
ball_samples = leukaemia_manifest_narrow[leukaemia_manifest_narrow['lineage'] == 'B-ALL']['sample']
allsorts_input_magici = counts_leukaemia_narrow[ball_samples].T
allsorts_input_magici.to_csv("data/allsorts_input_magici.csv")

print("ALLSorts input shape:", allsorts_input_magici.shape)

#### MRD data merge with MAGIC-I VST ####

In [ ]:
#attach study_id to full_df
full_df = tpmt_nudt15_vst.merge(manifest[['sample', 'study_id']], on='sample', how='left')

print(full_df['study_id'].isna().sum(), "samples missing study_id")
print(full_df[['sample', 'study_id']].head())

In [ ]:
mrd_df = pd.read_excel("data/MRD_MAGIC-I.xlsx")

full_df = full_df.merge(
    mrd_df,
    left_on='study_id',
    right_on='MAGIC-I inclusion',
    how='left'
)

print(f"Matched: {full_df['MAGIC-I inclusion'].notna().sum()} of {len(full_df)}")
print(full_df[['sample', 'study_id', 'Diag/Rel', 'Post-induction (Diag)', 'Post-Consolidation (Diag)', 'Post-induction (Relapse)']].head(10))

In [ ]:
def clean_mrd(val):
    if pd.isna(val):
        return None
    if isinstance(val, str):
        val_lower = val.strip().lower()
        if val_lower in ['not detected', 'nd']:
            return 0.0
        try:
            return float(val)
        except ValueError:
            return None  # catches non-numeric notes like "Blina"
    return val

full_df['mrd_value'] = full_df['Post-induction (Diag)'].apply(clean_mrd)

print(full_df[['sample', 'study_id', 'Post-induction (Diag)', 'mrd_value']].head(15))
print(f"\nUsable MRD values: {full_df['mrd_value'].notna().sum()} of {len(full_df)}")
print(f"Non-numeric/dropped entries: {full_df[full_df['Post-induction (Diag)'].notna() & full_df['mrd_value'].isna()]['Post-induction (Diag)'].unique()}")

This step: 
- converted "not detected"/"ND" -> 0.0 (undetectable MRD is treated as 0, consisted with how TARGET data was handled)
- tried to parse any other string as a float
- flagged anything that doesn't parse (e.g. "Blina" which seems lika  drug/treatment annotation and not an MRD value) 


In [ ]:
#generate status column using same standard threshold as TARGET (MRD pos ≥ 0.01%)
full_df['mrd_status'] = full_df['mrd_value'].apply(
    lambda x: 'positive' if pd.notna(x) and x >= 0.0001 else ('negative' if pd.notna(x) else None)
)

print(full_df[['sample', 'study_id', 'mrd_value', 'mrd_status']].head(15))
print(f"\nMRD status counts:\n{full_df['mrd_status'].value_counts(dropna=False)}")

In [ ]:
#exclude unmatched samples for MRD analyses
mrd_analysis_df = full_df[full_df['mrd_status'].notna()].copy()
print(f"Samples for MRD analysis: {len(mrd_analysis_df)}")
print(mrd_analysis_df['mrd_status'].value_counts())

In [ ]:
#Mann-Whitney U test - TPMT/NUDT15 vs MRD status (same as TARGET)
from scipy.stats import mannwhitneyu

pos = mrd_analysis_df[mrd_analysis_df['mrd_status'] == 'positive']
neg = mrd_analysis_df[mrd_analysis_df['mrd_status'] == 'negative']

tpmt_stat, tpmt_p = mannwhitneyu(pos['TPMT_vst'], neg['TPMT_vst'])
nudt15_stat, nudt15_p = mannwhitneyu(pos['NUDT15_vst'], neg['NUDT15_vst'])

print(f"TPMT vs MRD status (MAGIC-I): p={tpmt_p:.4f}")
print(f"  MRD+ mean={pos['TPMT_vst'].mean():.3f}, MRD- mean={neg['TPMT_vst'].mean():.3f}")
print(f"NUDT15 vs MRD status (MAGIC-I): p={nudt15_p:.4f}")
print(f"  MRD+ mean={pos['NUDT15_vst'].mean():.3f}, MRD- mean={neg['NUDT15_vst'].mean():.3f}")

In [ ]:
tpmt_pos_median = pos['TPMT_vst'].median()
tpmt_neg_median = neg['TPMT_vst'].median()
nudt15_pos_median = pos['NUDT15_vst'].median()
nudt15_neg_median = neg['NUDT15_vst'].median()

print(f"TPMT vs MRD status (MAGIC-I): p={tpmt_p:.4f}")
print(f"  MRD+ median={tpmt_pos_median:.3f}, MRD- median={tpmt_neg_median:.3f}")
print(f"NUDT15 vs MRD status (MAGIC-I): p={nudt15_p:.4f}")
print(f"  MRD+ median={nudt15_pos_median:.3f}, MRD- median={nudt15_neg_median:.3f}")

#### Differential Expression Analysis, MRD+ vs MRD- ####

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# Build count matrix restricted to samples with MRD status
mrd_samples = mrd_analysis_df['sample'].tolist()
counts_mrd = counts_leukaemia_narrow[mrd_samples].T  # samples as rows, genes as columns

# Metadata for DESeq2 design
metadata = mrd_analysis_df.set_index('sample').loc[mrd_samples, ['mrd_status']]

print(counts_mrd.shape)
print(metadata['mrd_status'].value_counts())

In [ ]:
dds = DeseqDataSet(
    counts=counts_mrd,
    metadata=metadata,
    design="~mrd_status",
    refit_cooks=True
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["mrd_status", "positive", "negative"])
stat_res.summary()

results_df_magic = stat_res.results_df

print(results_df_magic.shape)
print(results_df_magic.sort_values('padj').head(10))

In [ ]:
print(results_df_magic.loc['ENSG00000137364', ['log2FoldChange', 'pvalue', 'padj']])

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

all_ids = results_df_magic.index.tolist()
gene_map = mg.querymany(all_ids, scopes='ensembl.gene', fields='symbol', species='human', as_dataframe=True)

# build ensembl -> symbol mapping, dropping ones with no symbol found
id_to_symbol = gene_map['symbol'].dropna().to_dict()

results_df_symbols = results_df_magic.copy()
results_df_symbols['symbol'] = results_df_symbols.index.map(id_to_symbol)
results_df_symbols = results_df_symbols.dropna(subset=['symbol'])

# handle duplicate symbols (multiple Ensembl IDs mapping to same gene) - keep the one with highest |stat|
results_df_symbols['abs_stat'] = results_df_symbols['stat'].abs()
results_df_symbols = results_df_symbols.sort_values('abs_stat', ascending=False).drop_duplicates('symbol')

print(f"Original genes: {len(results_df_magic)}")
print(f"After symbol mapping and dedup: {len(results_df_symbols)}")

ranked = results_df_symbols.set_index('symbol')['stat'].sort_values(ascending=False).dropna()
ranked.index.name = None
print(ranked.head())

In [ ]:
import gseapy as gp

gsea_results = gp.prerank(
    rnk=ranked,
    gene_sets="KEGG_2021_Human",
    threads=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    outdir="gsea_output_magici",
    seed=42,
    verbose=False
)
gsea_df = gsea_results.res2d.sort_values("NES", ascending=False)
gsea_sig = gsea_df[gsea_df["FDR q-val"] < 0.25].copy()

gsea_hallmark = gp.prerank(
    rnk=ranked,
    gene_sets="MSigDB_Hallmark_2020",
    threads=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    outdir="gsea_hallmark_output_magici",
    seed=42,
    verbose=False
)
hallmark_df = gsea_hallmark.res2d.sort_values("NES", ascending=False)
hallmark_sig = hallmark_df[hallmark_df["FDR q-val"] < 0.25].copy()

print(gsea_sig[['Term','NES','FDR q-val']].head(20))
print()
print(hallmark_sig[['Term','NES','FDR q-val']].head(20))

In [ ]:
print("KEGG - Ribosome:")
print(gsea_df[gsea_df['Term'].str.contains('Ribosome', case=False, na=False)][['Term','NES','FDR q-val']])

print("\nKEGG - Glutathione:")
print(gsea_df[gsea_df['Term'].str.contains('Glutathione', case=False, na=False)][['Term','NES','FDR q-val']])

print("\nKEGG - Ferroptosis:")
print(gsea_df[gsea_df['Term'].str.contains('Ferroptosis', case=False, na=False)][['Term','NES','FDR q-val']])

print("\nHallmark - TNF-alpha:")
print(hallmark_df[hallmark_df['Term'].str.contains('TNF', case=False, na=False)][['Term','NES','FDR q-val']])

# get full gene lists from KEGG/Hallmark (not leading-edge, since only Ribosome was significant here)
kegg_lib = gp.get_library(name="KEGG_2021_Human")
hallmark_lib = gp.get_library(name="MSigDB_Hallmark_2020")

ribosome_genes = kegg_lib['Ribosome']
glutathione_genes = kegg_lib['Glutathione metabolism']
ferroptosis_genes = kegg_lib['Ferroptosis']
tnf_genes = hallmark_lib['TNF-alpha Signaling via NF-kB']

for name, genes in [('Ribosome', ribosome_genes), ('Glutathione', glutathione_genes),
                     ('Ferroptosis', ferroptosis_genes), ('TNF-alpha', tnf_genes)]:
    print(f"{name}: {len(genes)} genes in full canonical set")

In [ ]:
symbol_to_id = {v: k for k, v in id_to_symbol.items()}

def compute_pathway_score(gene_symbols, vst_data, symbol_to_id_map):
    ensembl_ids = [symbol_to_id_map[g] for g in gene_symbols if g in symbol_to_id_map]
    present = [g for g in ensembl_ids if g in vst_data.columns]
    sub = vst_data[present].T  # genes x samples
    z = sub.subtract(sub.mean(axis=1), axis=0).divide(sub.std(axis=1), axis=0)
    return z.mean(axis=0), len(present)

ribosome_score, n_ribo = compute_pathway_score(ribosome_genes, vst_df, symbol_to_id)
glutathione_score, n_glut = compute_pathway_score(glutathione_genes, vst_df, symbol_to_id)
ferroptosis_score, n_ferro = compute_pathway_score(ferroptosis_genes, vst_df, symbol_to_id)
tnf_score, n_tnf = compute_pathway_score(tnf_genes, vst_df, symbol_to_id)

print(f"Ribosome: {n_ribo}/{len(ribosome_genes)} genes found in vst_df")
print(f"Glutathione: {n_glut}/{len(glutathione_genes)} genes found")
print(f"Ferroptosis: {n_ferro}/{len(ferroptosis_genes)} genes found")
print(f"TNF-alpha: {n_tnf}/{len(tnf_genes)} genes found")
# merge all four scores into one dataframe
pathway_scores = pd.DataFrame({
    'ribosome_score': ribosome_score,
    'glutathione_score': glutathione_score,
    'ferroptosis_score': ferroptosis_score,
    'tnf_score': tnf_score
})
pathway_scores.index.name = 'sample'
pathway_scores = pathway_scores.reset_index()

# merge with lineage + TPMT/NUDT15
full_df = full_df.merge(pathway_scores, on='sample')
full_df = full_df.merge(lib_sizes.rename('lib_size').reset_index().rename(columns={'index':'sample'}), on='sample')

print(full_df.head())

# library-size correlation check (same as TARGET diagnostic)
print("\nLibrary size correlations:")
for col in ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']:
    rho, p = spearmanr(full_df['lib_size'], full_df[col])
    print(f"{col}: rho={rho:.3f}, p={p:.4f}")

In [ ]:
# lineage comparison (B-ALL vs T-ALL only, adequate n)
ball = full_df[full_df['lineage']=='B-ALL']
tall = full_df[full_df['lineage']=='T-ALL']

print("Pathway scores by lineage (B-ALL vs T-ALL):")
for col in ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']:
    stat, p = stats.mannwhitneyu(ball[col], tall[col])
    print(f"{col}: B-ALL mean={ball[col].mean():.3f}, T-ALL mean={tall[col].mean():.3f}, p={p:.4f}")

# correlation with TPMT/NUDT15 across all samples
print("\nTPMT/NUDT15 vs pathway scores:")
for col in ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']:
    r_t, p_t = spearmanr(full_df['TPMT_vst'], full_df[col])
    r_n, p_n = spearmanr(full_df['NUDT15_vst'], full_df[col])
    print(f"{col} vs TPMT: rho={r_t:.3f}, p={p_t:.4f}  |  vs NUDT15: rho={r_n:.3f}, p={p_n:.4f}")

In [ ]:
# lineage comparison (B-ALL vs T-ALL only, adequate n)
ball = full_df[full_df['lineage']=='B-ALL']
tall = full_df[full_df['lineage']=='T-ALL']

print("Pathway scores by lineage (B-ALL vs T-ALL):")
for col in ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']:
    stat, p = stats.mannwhitneyu(ball[col], tall[col])
    print(f"{col}: B-ALL mean={ball[col].mean():.3f}, T-ALL mean={tall[col].mean():.3f}, p={p:.4f}")

# correlation with TPMT/NUDT15 across all samples
print("\nTPMT/NUDT15 vs pathway scores:")
for col in ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']:
    r_t, p_t = spearmanr(full_df['TPMT_vst'], full_df[col])
    r_n, p_n = spearmanr(full_df['NUDT15_vst'], full_df[col])
    print(f"{col} vs TPMT: rho={r_t:.3f}, p={p_t:.4f}  |  vs NUDT15: rho={r_n:.3f}, p={p_n:.4f}")
    
# check if TPMT-ribosome correlation holds within B-ALL alone (largest group, n=32)
r_ball, p_ball = spearmanr(ball['TPMT_vst'], ball['ribosome_score'])
print(f"Within B-ALL only: TPMT vs ribosome rho={r_ball:.3f}, p={p_ball:.4f}")

r_ball_ferro, p_ball_ferro = spearmanr(ball['TPMT_vst'], ball['ferroptosis_score'])
print(f"Within B-ALL only: TPMT vs ferroptosis rho={r_ball_ferro:.3f}, p={p_ball_ferro:.4f}")

check_cols = ['Infiltration', 'Sex', 'Disease Category', 'SNOMED CT', 'PMC /RNAseq', 'PMC WES']
for col in check_cols:
    print(f"=== {col} ===")
    print(merged[col].value_counts(dropna=False).head(10))
    print()

sex_df = merged[['sample', 'Sex']].merge(full_df.drop(columns=['Sex'], errors='ignore'), on='sample')

print("Sex distribution:", sex_df['Sex'].value_counts().to_dict())

test_cols = ['TPMT_vst', 'NUDT15_vst', 'ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']
for col in test_cols:
    m_vals = sex_df[sex_df['Sex']=='M'][col]
    f_vals = sex_df[sex_df['Sex']=='F'][col]
    stat, p = stats.mannwhitneyu(m_vals, f_vals)
    print(f"{col}: M mean={m_vals.mean():.3f}, F mean={f_vals.mean():.3f}, p={p:.4f}")

import numpy as np

def clean_disease_category(x):
    if pd.isna(x):
        return np.nan
    x = x.strip()
    if x.startswith('HH') or x == 'High Hyperdiploidy':
        return 'High Hyperdiploidy'
    return x

merged['disease_cat_clean'] = merged['Disease Category'].apply(clean_disease_category)
print(merged['disease_cat_clean'].value_counts(dropna=False))

# High Hyperdiploidy vs all other known categories (excluding NaN/untested)
hh_df = merged[['sample', 'disease_cat_clean']].merge(full_df, on='sample')
hh_df = hh_df.dropna(subset=['disease_cat_clean'])
hh_df['is_HH'] = hh_df['disease_cat_clean'] == 'High Hyperdiploidy'

print(f"\nHH: {hh_df['is_HH'].sum()}, non-HH (known category): {(~hh_df['is_HH']).sum()}")

test_cols = ['TPMT_vst', 'NUDT15_vst', 'ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']
for col in test_cols:
    hh_vals = hh_df[hh_df['is_HH']][col]
    other_vals = hh_df[~hh_df['is_HH']][col]
    stat, p = stats.mannwhitneyu(hh_vals, other_vals)
    print(f"{col}: HH mean={hh_vals.mean():.3f}, other mean={other_vals.mean():.3f}, p={p:.4f}")

In [ ]:
subtype_markers = {
    'ETV6::RUNX1': ['ETV6', 'RUNX1', 'SPIB', 'MME'],
    'TCF3::PBX1': ['PBX1', 'TCF3', 'MEIS1'],
    'High Hyperdiploid': ['CRLF2', 'IL3RA', 'FLT3', 'CD200'],
    'DUX4-rearranged': ['DUX4', 'ERG', 'CD2'],
    'PAX5alt': ['PAX5', 'BLNK', 'VPREB1'],
    'Ph-like': ['CRLF2', 'IL7R', 'JAK2', 'EPOR'],
    'KMT2A-rearranged': ['KMT2A', 'MEIS1', 'FLT3', 'PROM1'],
}

subtype_marker_ids = {
    subtype: [symbol_to_id[g] for g in genes if g in symbol_to_id]
    for subtype, genes in subtype_markers.items()
}
for subtype, ids in subtype_marker_ids.items():
    print(f"{subtype}: {len(ids)}/{len(subtype_markers[subtype])} genes found")

In [ ]:
def compute_subtype_score(gene_ids, vst_data):
    present = [g for g in gene_ids if g in vst_data.columns]
    sub = vst_data[present].T
    z = sub.subtract(sub.mean(axis=1), axis=0).divide(sub.std(axis=1), axis=0)
    return z.mean(axis=0)

subtype_scores = pd.DataFrame({
    subtype: compute_subtype_score(ids, vst_df)
    for subtype, ids in subtype_marker_ids.items()
})
subtype_scores.index.name = 'sample'

# restrict to B-ALL samples only
ball_samples_list = leukaemia_manifest_strict[leukaemia_manifest_strict['lineage']=='B-ALL']['sample']
subtype_scores_ball = subtype_scores.loc[subtype_scores.index.isin(ball_samples_list)]

# assign each sample its top-scoring subtype
subtype_scores_ball = subtype_scores_ball.copy()
subtype_scores_ball['assigned_subtype'] = subtype_scores_ball.idxmax(axis=1)
subtype_scores_ball['top_score'] = subtype_scores_ball.drop(columns=['assigned_subtype'], errors='ignore').max(axis=1)

print(subtype_scores_ball[['assigned_subtype', 'top_score']].sort_values('assigned_subtype'))
print()
print(subtype_scores_ball['assigned_subtype'].value_counts())

In [ ]:
t_all_markers = {
    'TLX1': ['TLX1', 'BCL2'],
    'TLX3': ['TLX3', 'NKX3-1'],
    'TAL1/LMO': ['TAL1', 'LMO1', 'LMO2', 'LYL1'],
    'HOXA': ['HOXA9', 'HOXA10', 'MEIS1'],
    'NKX2-1': ['NKX2-1', 'NKX2-2'],
    'Immature/ETP': ['LYL1', 'LMO2', 'CD34', 'KIT'],
    'MEF2C': ['MEF2C', 'DNTT'],
}

t_all_marker_ids = {
    subtype: [symbol_to_id[g] for g in genes if g in symbol_to_id]
    for subtype, genes in t_all_markers.items()
}
for subtype, ids in t_all_marker_ids.items():
    print(f"{subtype}: {len(ids)}/{len(t_all_markers[subtype])} genes found")

In [ ]:
def compute_subtype_score(gene_ids, vst_data):
    present = [g for g in gene_ids if g in vst_data.columns]
    sub = vst_data[present].T
    z = sub.subtract(sub.mean(axis=1), axis=0).divide(sub.std(axis=1), axis=0)
    return z.mean(axis=0)

t_all_scores = pd.DataFrame({
    subtype: compute_subtype_score(ids, vst_df)
    for subtype, ids in t_all_marker_ids.items()
})
t_all_scores.index.name = 'sample'

# restrict to T-ALL samples only
tall_samples_list = leukaemia_manifest_strict[leukaemia_manifest_strict['lineage']=='T-ALL']['sample']
t_all_scores_tall = t_all_scores.loc[t_all_scores.index.isin(tall_samples_list)]

t_all_scores_tall = t_all_scores_tall.copy()
t_all_scores_tall['assigned_subtype'] = t_all_scores_tall.idxmax(axis=1)
t_all_scores_tall['top_score'] = t_all_scores_tall.drop(columns=['assigned_subtype'], errors='ignore').max(axis=1)

print(t_all_scores_tall[['assigned_subtype', 'top_score']].sort_values('assigned_subtype'))
print()
print(t_all_scores_tall['assigned_subtype'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- Plot 1: TPMT/NUDT15 by lineage (bar + error bars) ----
lineage_order = ['B-ALL', 'T-ALL', 'MPAL']
summary = tpmt_nudt15_vst_strict.groupby('lineage')[['TPMT_vst', 'NUDT15_vst']].agg(['mean', 'std', 'count'])
summary = summary.loc[lineage_order]

x = np.arange(len(lineage_order))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
tpmt_means = summary[('TPMT_vst', 'mean')].values
tpmt_stds = summary[('TPMT_vst', 'std')].fillna(0).values
nudt15_means = summary[('NUDT15_vst', 'mean')].values
nudt15_stds = summary[('NUDT15_vst', 'std')].fillna(0).values

ax.bar(x - width/2, tpmt_means, width, yerr=tpmt_stds, capsize=4, label='TPMT', color='#2a78d6')
ax.bar(x + width/2, nudt15_means, width, yerr=nudt15_stds, capsize=4, label='NUDT15', color='#eb6834')

labels = [f"{l} (n={int(summary.loc[l, ('TPMT_vst','count')])})" for l in lineage_order]
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('VST-normalized expression')
ax.set_title('TPMT and NUDT15 expression by leukaemia lineage')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('tpmt_nudt15_by_lineage.png', dpi=300)
plt.show()

# ---- Plot 2: TPMT vs Ribosome pathway score, colored by lineage ----
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'B-ALL': '#2a78d6', 'T-ALL': '#eb6834', 'AML': '#1baf7a', 'MPAL': '#eda100'}

for lin in lineage_order:
    sub = full_df[full_df['lineage'] == lin]
    ax.scatter(sub['ribosome_score'], sub['TPMT_vst'], label=lin, color=colors[lin], s=50, alpha=0.85)

ax.set_xlabel('Ribosome pathway score')
ax.set_ylabel('TPMT (VST-normalized)')
ax.set_title('TPMT expression vs ribosome pathway score')
ax.legend(title='Lineage')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('tpmt_vs_ribosome_score.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# Build count matrix restricted to samples with MRD status
mrd_samples = mrd_analysis_df['sample'].tolist()
counts_mrd = counts_leukaemia_narrow[mrd_samples].T  # samples as rows, genes as columns

# Metadata for DESeq2 design
metadata = mrd_analysis_df.set_index('sample').loc[mrd_samples, ['mrd_status']]

print(counts_mrd.shape)
print(metadata['mrd_status'].value_counts()) 
dds = DeseqDataSet(
    counts=counts_mrd,
    metadata=metadata,
    design="~mrd_status",
    refit_cooks=True
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["mrd_status", "positive", "negative"])
stat_res.summary()
results_df_magic = stat_res.results_df

print(results_df_magic.shape)
print(results_df_magic.sort_values('padj').head(10))

df = results_df_magic.dropna(subset=['padj', 'log2FoldChange']).copy()
df['neg_log10_padj'] = -np.log10(df['padj'].replace(0, 1e-300))

sig = (df['padj'] < 0.05) & (df['log2FoldChange'].abs() > 1)
up = sig & (df['log2FoldChange'] > 0)
down = sig & (df['log2FoldChange'] < 0)
notsig = ~sig

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df.loc[notsig, 'log2FoldChange'], df.loc[notsig, 'neg_log10_padj'],
           s=4, color='#c3c2b7', alpha=0.5, label='Not significant')
ax.scatter(df.loc[down, 'log2FoldChange'], df.loc[down, 'neg_log10_padj'],
           s=4, color='#378ADD', alpha=0.6, label='Downregulated (higher in MRD-)')
ax.scatter(df.loc[up, 'log2FoldChange'], df.loc[up, 'neg_log10_padj'],
           s=4, color='#E24B4A', alpha=0.6, label='Upregulated (higher in MRD+)')

ax.set_ylim(top=df['neg_log10_padj'].max() * 1.23)

sig_genes = df[sig].sort_values('padj')
top_genes = sig_genes.head(10).index.tolist()

# manual (dx, dy) offsets in points, tuned to fan out the crowded cluster
manual_offsets = [
    (5, 5),     # ALDH1A2
    (-70, 15),  # LOC124905143
    (-70, -5),  # ARHGEF4
    (-60, 15),  # gene 4 (was ELOVL4 area)
    (30, 25),   # gene 5 (was CCR9 area)
    (40, -15),  # gene 6 (NDST3 area)
    (-90, -15), # gene 7 (MYO7B)
    (30, -20),  # gene 8 (GCSAML)
    (-90, -25), # gene 9 (KRT73)
    (10, -35),  # gene 10 (RFLNA)
]

for gid, (dx, dy) in zip(top_genes, manual_offsets):
    if gid in df.index and gid in id_to_symbol:
        row = df.loc[gid]
        ax.annotate(id_to_symbol[gid], (row['log2FoldChange'], row['neg_log10_padj']),
                    fontsize=8, xytext=(dx, dy), textcoords='offset points',
                    arrowprops=dict(arrowstyle='-', color='#898781', lw=0.5))

ax.axhline(-np.log10(0.05), color='#898781', linestyle='--', linewidth=0.8)
ax.axvline(1, color='#898781', linestyle='--', linewidth=0.8)
ax.axvline(-1, color='#898781', linestyle='--', linewidth=0.8)
ax.set_xlabel('log2 fold change (MRD+ vs MRD-)')
ax.set_ylabel('-log10(adjusted p-value)')
ax.set_title('Differential expression: MRD+ vs MRD- (MAGIC-I)')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('volcano_mrd_pos_neg.png', dpi=300)
plt.show()

In [ ]:
# Map Ensembl IDs to gene symbols
de_table = results_df_magic.copy()
de_table['gene_symbol'] = de_table.index.map(id_to_symbol)

# Drop genes with no symbol mapping, keep only relevant columns
de_table = de_table.dropna(subset=['gene_symbol'])
de_table = de_table[['gene_symbol', 'baseMean', 'log2FoldChange', 'pvalue', 'padj']]

# Sort by adjusted p-value, most significant first
de_table = de_table.sort_values('padj')

print(de_table.head(20))

# Save full table to CSV
de_table.to_csv('magic_i_de_results_mrd.csv', index=False)

In [ ]:
print(results_df_magic.loc['ENSG00000137364'])  # TPMT
print(results_df_magic.loc['ENSG00000136159'])  # NUDT15

In [ ]:
# Top annotated (gene symbol available) DE genes, matching what's labeled on the volcano plot
top_annotated = de_table.dropna(subset=['gene_symbol']).sort_values('padj').head(10)

# TPMT and NUDT15 rows (included regardless of significance, for reference)
gene_ids = {
    'TPMT': 'ENSG00000137364',
    'NUDT15': 'ENSG00000136159'
}
tpmt_nudt15_rows = de_table[de_table.index.isin(gene_ids.values())].copy()

# Combine and drop duplicates in case TPMT/NUDT15 are already in the top 10
final_table = pd.concat([top_annotated, tpmt_nudt15_rows]).drop_duplicates()
final_table = final_table.sort_values('padj')

print(final_table)

final_table.to_csv('magic_i_top_de_genes_plus_tpmt_nudt15.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

top_kegg = gsea_df.sort_values('NES', ascending=False).head(10)
top_hallmark = hallmark_df.sort_values('NES', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, data, title in [(axes[0], top_kegg, 'Top KEGG pathways (T-ALL vs B-ALL)'),
                          (axes[1], top_hallmark, 'Top Hallmark pathways (T-ALL vs B-ALL)')]:
    colors = ['#E24B4A' if fdr < 0.05 else '#c3c2b7' for fdr in data['FDR q-val']]
    ax.barh(data['Term'], data['NES'], color=colors)
    ax.set_xlabel('Normalized enrichment score (NES)')
    ax.set_title(title, fontsize=11)
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('gsea_top_pathways.png', dpi=300)
plt.show()

In [ ]:
import numpy as np

genes = ['TPMT_vst', 'NUDT15_vst']
pathways = ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score']

corr_matrix = np.zeros((len(genes), len(pathways)))
pval_matrix = np.zeros((len(genes), len(pathways)))

for i, g in enumerate(genes):
    for j, p in enumerate(pathways):
        r, pv = spearmanr(full_df[g], full_df[p])
        corr_matrix[i, j] = r
        pval_matrix[i, j] = pv

fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')

ax.set_xticks(range(len(pathways)))
ax.set_xticklabels(['Ribosome', 'Glutathione', 'Ferroptosis', 'TNF-alpha'])
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(['TPMT', 'NUDT15'])

for i in range(len(genes)):
    for j in range(len(pathways)):
        sig_marker = '*' if pval_matrix[i, j] < 0.05 else ''
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}{sig_marker}", ha='center', va='center',
                color='white' if abs(corr_matrix[i,j]) > 0.35 else 'black', fontsize=11)

plt.colorbar(im, ax=ax, label="Spearman's rho", shrink=0.8)
ax.set_title('TPMT/NUDT15 correlation with pathway scores (* = p<0.05)', fontsize=11)
plt.tight_layout()
plt.savefig('tpmt_nudt15_pathway_heatmap.png', dpi=300)
plt.show()

In [ ]:
extra_pathways = {
    'Purine metabolism': ('kegg', 'Purine metabolism'),
    'Apoptosis': ('kegg', 'Apoptosis'),
    'Oxidative phosphorylation': ('kegg', 'Oxidative phosphorylation'),
    'mTORC1 signaling': ('hallmark', 'mTORC1 Signaling'),
    'p53 pathway': ('hallmark', 'p53 Pathway'),
}

extra_scores = {}
for label, (source, term) in extra_pathways.items():
    lib = kegg_lib if source == 'kegg' else hallmark_lib
    if term in lib:
        genes = lib[term]
        score, n_found = compute_pathway_score(genes, vst_df, symbol_to_id)
        extra_scores[label] = score
        print(f"{label}: {n_found}/{len(genes)} genes found")
    else:
        print(f"{label}: term '{term}' not found in {source} library")

# merge into full_df
for label, score in extra_scores.items():
    full_df[label.lower().replace(' ', '_') + '_score'] = full_df['sample'].map(score)

In [ ]:
pathways_extended = ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score'] + \
                     [label.lower().replace(' ', '_') + '_score' for label in extra_pathways]
pathway_labels = ['Ribosome', 'Glutathione', 'Ferroptosis', 'TNF-alpha'] + list(extra_pathways.keys())

corr_matrix = np.zeros((2, len(pathways_extended)))
pval_matrix = np.zeros((2, len(pathways_extended)))
for i, g in enumerate(['TPMT_vst', 'NUDT15_vst']):
    for j, p in enumerate(pathways_extended):
        r, pv = spearmanr(full_df[g], full_df[p])
        corr_matrix[i, j] = r
        pval_matrix[i, j] = pv

fig, ax = plt.subplots(figsize=(2 + len(pathways_extended)*1.3, 3.5))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')
ax.set_xticks(range(len(pathways_extended)))
ax.set_xticklabels(pathway_labels, rotation=30, ha='right')
ax.set_yticks([0, 1])
ax.set_yticklabels(['TPMT', 'NUDT15'])
for i in range(2):
    for j in range(len(pathways_extended)):
        marker = '*' if pval_matrix[i, j] < 0.05 else ''
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}{marker}", ha='center', va='center',
                color='white' if abs(corr_matrix[i,j]) > 0.35 else 'black', fontsize=10)
plt.colorbar(im, ax=ax, label="Spearman's rho", shrink=0.8)
plt.tight_layout()
plt.savefig('tpmt_nudt15_pathway_heatmap_extended.png', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
colors = {'B-ALL': '#2a78d6', 'T-ALL': '#eb6834', 'AML': '#1baf7a', 'MPAL': '#eda100'}
lineage_order = ['B-ALL', 'T-ALL', 'AML', 'MPAL']

for ax, score_col, title in [(axes[0], 'ferroptosis_score', 'TPMT vs Ferroptosis score'),
                               (axes[1], 'tnf_score', 'TPMT vs TNF-alpha score')]:
    for lin in lineage_order:
        sub = full_df[full_df['lineage'] == lin]
        ax.scatter(sub[score_col], sub['TPMT_vst'], label=lin, color=colors[lin], s=55, alpha=0.85)
    ax.set_xlabel(title.split(' vs ')[1])
    ax.set_ylabel('TPMT (VST-normalized)')
    ax.set_title(title)
    ax.spines[['top', 'right']].set_visible(False)

axes[1].legend(title='Lineage', loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig('tpmt_ferroptosis_tnf_scatter.png', dpi=300)
plt.show()

In [ ]:
top_gene = 'ENSG00000128918'
raw_vals = counts_mrd[top_gene]
summary_by_status = pd.concat([raw_vals, metadata['mrd_status']], axis=1).groupby('mrd_status')[top_gene].describe()
print(summary_by_status)


In [ ]:
for gene in results_df_magic.sort_values('padj').head(10).index:
    vals = counts_mrd[gene]
    print(f"{gene}: max={vals.max():.0f}, median={vals.median():.0f}, ratio={vals.max()/max(vals.median(),1):.0f}x")

    # Build the outlier ratio table
outlier_rows = []
for gene in results_df_magic.sort_values('padj').head(10).index:
    vals = counts_mrd[gene]
    outlier_rows.append({
        'gene_id': gene,
        'gene_symbol': id_to_symbol.get(gene, gene),
        'max_count': vals.max(),
        'median_count': vals.median(),
        'ratio': vals.max() / max(vals.median(), 1)
    })

outlier_df = pd.DataFrame(outlier_rows)
print(outlier_df)

outlier_df.to_csv('magic_i_top10_outlier_ratios.csv', index=False)

In [ ]:
print(f"({len(outlier_samples)} samples — see saved outlier table for detail)")


In [ ]:
import mygene
mg = mygene.MyGeneInfo()

top_gene_id = results_df_magic.sort_values('padj').index[0]
print(f"Current top gene by padj: {top_gene_id}")

result = mg.getgene(top_gene_id, fields='symbol,name')
print(result)

In [ ]:
# Check subtype/fusion composition of MRD+ vs MRD- groups
print(merged[merged['sample'].isin(mrd_analysis_df['sample'])][['sample', 'Fusion']].merge(
    mrd_analysis_df[['sample', 'mrd_status']], on='sample'
).groupby('mrd_status')['Fusion'].value_counts())

In [ ]:
outlier_samples_list = list(outlier_samples)
clean_samples = [s for s in mrd_analysis_df['sample'] if s not in outlier_samples_list]

clean_metadata = metadata.loc[clean_samples]
print(clean_metadata['mrd_status'].value_counts())

In [ ]:
metadata['fusion_status'] = merged.set_index('sample').loc[metadata.index, 'Fusion'].apply(
    lambda x: 'fusion_negative' if pd.isna(x) or 'negative' in str(x).lower() else 'fusion_positive'
)
print(metadata['fusion_status'].value_counts())

In [ ]:
metadata['fusion_status'] = merged.set_index('sample').loc[metadata.index, 'Fusion'].apply(
    lambda x: 'fusion_negative' if pd.isna(x) or 'negative' in str(x).lower() else 'fusion_positive'
)

dds_adj = DeseqDataSet(counts=counts_mrd, metadata=metadata, design="~fusion_status + mrd_status")
dds_adj.deseq2()

stat_res_adj = DeseqStats(dds_adj, contrast=["mrd_status", "positive", "negative"])
stat_res_adj.summary()

results_adj = stat_res_adj.results_df
print(results_adj.sort_values('padj').head(10))

#### Pathway module scores vs MRD status ####

In [ ]:
from scipy.stats import mannwhitneyu

pathway_cols = [
    'ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score',
    'purine_metabolism_score', 'apoptosis_score', 'oxidative_phosphorylation_score', 'mtorc1_signaling_score', 'p53_pathway_score'
]

mrd_pathway_df = full_df[full_df['mrd_status'].notna()].copy()
pos = mrd_pathway_df[mrd_pathway_df['mrd_status'] == 'positive']
neg = mrd_pathway_df[mrd_pathway_df['mrd_status'] == 'negative']

for col in pathway_cols:
    stat, p = mannwhitneyu(pos[col], neg[col])
    print(f"{col}: p={p:.4f}, MRD+ median={pos[col].median():.3f}, MRD- median={neg[col].median():.3f}")

Used four KEGG pathways from TARGET analysis and five from Hallmark gene sets.
None of the ten pathways reach significance (lowest is apoptosis, p=0.06). At the pathway-level, MAGIC-I shows no MRD-associated signal across any of the ten TARGET-matched pathways. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_labels = {
    'ribosome_score': 'Ribosome',
    'glutathione_score': 'Glutathione',
    'ferroptosis_score': 'Ferroptosis',
    'tnf_score': 'TNF signaling',
    'purine_metabolism_score': 'Purine metab.',
    'apoptosis_score': 'Apoptosis',
    'oxidative_phosphorylation_score': 'Ox. phosphorylation',
    'mtorc1_signaling_score': 'mTORC1',
    'p53_pathway_score': 'p53 pathway'
}

# Reshape to long format for seaborn
plot_df = mrd_pathway_df[['mrd_status'] + pathway_cols].melt(
    id_vars='mrd_status', var_name='pathway', value_name='score'
)
plot_df['pathway'] = plot_df['pathway'].map(plot_labels)

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=plot_df, x='pathway', y='score', hue='mrd_status',
    palette={'positive': '#2a78d6', 'negative': '#eb6834'},
    ax=ax
)
ax.set_xlabel('')
ax.set_ylabel('Pathway module score')
ax.set_title('MAGIC-I pathway scores by MRD status (n=43)')
plt.xticks(rotation=45, ha='right')
ax.spines[['top', 'right']].set_visible(False)
plt.legend(title='MRD status', loc='upper right')
plt.tight_layout()
plt.savefig('magic_i_pathway_boxplot.png', dpi=300)
plt.show()

#### GSEA ####

In [ ]:
import gseapy as gp

# Rank genes by Wald statistic (same approach as TARGET)
ranked_genes = results_df_magic.dropna(subset=['stat']).sort_values('stat', ascending=False)

# Map Ensembl IDs to gene symbols for GSEA (KEGG/Hallmark libraries use symbols)
ranking = ranked_genes['stat']
ranking.index = ranking.index.map(lambda x: symbol_to_id.get(x, x))  
print(ranking.head())
print(ranking.shape)

In [ ]:
id_to_symbol = {v: k for k, v in symbol_to_id.items()}

ranking.index = ranked_genes.index.map(lambda x: id_to_symbol.get(x, x))
print(ranking.head())
print(f"Mapped: {sum(ranking.index != ranked_genes.index)} of {len(ranking)}")

75% mapping rate - enough coverage to run GSEA reliably 


In [ ]:
gsea_results_magic = gp.prerank(
    rnk=ranking,
    gene_sets=['KEGG_2021_Human', 'MSigDB_Hallmark_2020'],
    permutation_num=1000,
    seed=42,
    outdir=None
)

print(gsea_results_magic.res2d.sort_values('NES', ascending=False).head(15)[['Term', 'NES', 'FDR q-val']])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = [
    ('T cell receptor signaling', 2.4749, 'KEGG'),
    ('PD-L1/PD-1 checkpoint', 2.3638, 'KEGG'),
    ('NK cell mediated cytotoxicity', 2.2940, 'KEGG'),
    ('Allograft rejection', 2.1192, 'Hallmark'),
    ('Th17 cell differentiation', 2.1067, 'KEGG'),
    ('Measles', 2.0158, 'KEGG'),
    ('Steroid biosynthesis', 1.9933, 'KEGG'),
    ('Lysosome', 1.9711, 'KEGG'),
    ('IL-6/JAK/STAT3 signaling', 1.9618, 'Hallmark'),
    ('Th1 and Th2 differentiation', 1.9413, 'KEGG'),
    ('Yersinia infection', 1.9235, 'KEGG'),
    ('Inflammatory response', 1.8926, 'Hallmark'),
    ('TNF-alpha signaling via NF-kB', 1.8626, 'Hallmark'),
    ('IL-2/STAT5 signaling', 1.8585, 'Hallmark'),
    ('Acute myeloid leukemia', 1.8576, 'KEGG'),
]

labels = [d[0] for d in data][::-1]
nes = [d[1] for d in data][::-1]
sources = [d[2] for d in data][::-1]
colors = ['#2a78d6' if s == 'KEGG' else '#eb6834' for s in sources]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(labels, nes, color=colors)
ax.set_xlabel('Normalized enrichment score (NES)')
ax.set_title('Top 15 GSEA pathways — MAGIC-I MRD+ vs MRD- (n=43)')
ax.spines[['top', 'right']].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2a78d6', label='KEGG 2021 Human'),
    Patch(facecolor='#eb6834', label='MSigDB Hallmark 2020')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('magic_i_gsea_top15.png', dpi=300)
plt.show()

In [ ]:
from scipy.stats import spearmanr

print("Library size correlations (MAGIC-I):")
for col in pathway_cols:
    rho, p = spearmanr(full_df['lib_size'], full_df[col])
    print(f"{col}: rho={rho:.3f}, p={p:.4f}")

#### Multivariate regression - TPMT ~ mrd_status + fusion_status on MAGIC-I ####

In [ ]:
import statsmodels.formula.api as smf

reg_df = mrd_pathway_df.merge(
    merged[['sample', 'Fusion']].assign(
        fusion_status=lambda d: d['Fusion'].apply(
            lambda x: 'fusion_negative' if pd.isna(x) or 'negative' in str(x).lower() else 'fusion_positive'
        )
    )[['sample', 'fusion_status']],
    on='sample', how='left'
)

model = smf.ols('TPMT_vst ~ mrd_status + fusion_status', data=reg_df).fit()
print(model.summary())

In [ ]:
coefs = model.params.drop('Intercept')
conf_int = model.conf_int().drop('Intercept')
pvals = model.pvalues.drop('Intercept')

labels = ['MRD+ status', 'Fusion positive']

fig, ax = plt.subplots(figsize=(7, 4))

y_pos = np.arange(len(coefs))

ax.errorbar(
    coefs.values, y_pos,
    xerr=[coefs.values - conf_int[0].values, conf_int[1].values - coefs.values],
    fmt='o', color='#1E2761', ecolor='#028090', elinewidth=2, capsize=4, markersize=8
)

ax.axvline(0, color='grey', linestyle='--', linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel('Coefficient (95% CI)', fontsize=12)
ax.set_title('TPMT expression: fusion-adjusted regression coefficients (MAGIC-I, n=41)', fontsize=12)

for i, (c, p) in enumerate(zip(coefs.values, pvals.values)):
    offset = 0.15 if i == 0 else -0.25
    va = 'bottom' if i == 0 else 'top'
    ax.text(c, i + offset, f'p={p:.3f}', ha='center', va=va, fontsize=10, color='#444')
    
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('tpmt_regression_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()

**MRD+ status:** dot sits almost exactly at 0, and line stretches from -0.55 to +0.48. Visually confirms non-significnance (p=0.909). There is no detectable relationship between MRD status and TPMT expression in this cohort. 

**Fusion positive**: dot sits further from zero (~0.38, suggesting fusion-positive patients might have somewhat higher TPMT), but confidence interval still stretches from about -0.13 to +0.88. Not statistically significant (p=0.139).

In [ ]:
import matplotlib.pyplot as plt

coef_table = model.summary2().tables[1].round(4)
coef_table = coef_table.reset_index().rename(columns={'index': 'Term'})

# Shorten the long term labels so they fit
coef_table['Term'] = coef_table['Term'].replace({
    'mrd_status[T.positive]': 'MRD status (positive)',
    'fusion_status[T.fusion_positive]': 'Fusion status (positive)'
})

fig, ax = plt.subplots(figsize=(9, 2.2))
ax.axis('off')
tbl = ax.table(cellText=coef_table.values, colLabels=coef_table.columns,
               cellLoc='center', loc='center',
               colWidths=[0.28, 0.12, 0.12, 0.1, 0.1, 0.14, 0.14])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
plt.title('TPMT_vst ~ mrd_status + fusion_status (OLS)', pad=20)
plt.tight_layout()
plt.savefig('tpmt_ols_table.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
model_nudt15 = smf.ols('NUDT15_vst ~ mrd_status + fusion_status', data=reg_df).fit()
print(model_nudt15.summary())

In [ ]:
coefs = model_nudt15.params.drop('Intercept')
conf_int = model_nudt15.conf_int().drop('Intercept')
pvals = model_nudt15.pvalues.drop('Intercept')

labels = ['MRD+ status', 'Fusion positive']

fig, ax = plt.subplots(figsize=(7, 4))

y_pos = np.arange(len(coefs))

ax.errorbar(
    coefs.values, y_pos,
    xerr=[coefs.values - conf_int[0].values, conf_int[1].values - coefs.values],
    fmt='o', color='#1E2761', ecolor='#028090', elinewidth=2, capsize=4, markersize=8
)

ax.axvline(0, color='grey', linestyle='--', linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel('Coefficient (95% CI)', fontsize=12)
ax.set_title('NUDT15 expression: fusion-adjusted regression coefficients (MAGIC-I, n=41)', fontsize=12)

for i, (c, p) in enumerate(zip(coefs.values, pvals.values)):
    offset = 0.15 if i == 0 else -0.25
    va = 'bottom' if i == 0 else 'top'
    ax.text(c, i + offset, f'p={p:.3f}', ha='center', va=va, fontsize=10, color='#444')

ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('nudt15_regression_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

coef_table_nudt15 = model_nudt15.summary2().tables[1].round(4)
coef_table_nudt15 = coef_table_nudt15.reset_index().rename(columns={'index': 'Term'})

coef_table_nudt15['Term'] = coef_table_nudt15['Term'].replace({
    'mrd_status[T.positive]': 'MRD status (positive)',
    'fusion_status[T.fusion_positive]': 'Fusion status (positive)'
})

fig, ax = plt.subplots(figsize=(9, 2.2))
ax.axis('off')
tbl = ax.table(cellText=coef_table_nudt15.values, colLabels=coef_table_nudt15.columns,
               cellLoc='center', loc='center',
               colWidths=[0.28, 0.12, 0.12, 0.1, 0.1, 0.14, 0.14])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
plt.title('NUDT15_vst ~ mrd_status + fusion_status (OLS)', pad=20)
plt.tight_layout()
plt.savefig('nudt15_ols_table.png', dpi=300, bbox_inches='tight')
plt.show()

#### Genotype Analysis ####

In [ ]:
print(variant_df.shape)
print(variant_df.columns.tolist())


In [ ]:
variant_clean = variant_df[variant_df['Filter'] == 'PASS'].copy()

print(variant_clean['Sample'].nunique(), 'unique patients')
print(variant_clean['Gene'].value_counts())
print(variant_clean[['Chr','Pos']].drop_duplicates().shape[0], 'unique variant positions')

In [ ]:
tpmt_calls = call_star_alleles(variant_clean, 'TPMT', tpmt_no_function)
nudt15_calls = call_star_alleles(variant_clean, 'NUDT15', nudt15_no_function)

from collections import Counter
print("TPMT calls (per-patient counts):", Counter(tuple(a) for a in tpmt_calls.values()))
print("NUDT15 calls (per-patient counts):", Counter(tuple(a) for a in nudt15_calls.values()))


In [ ]:
tpmt_priority = ['*3A', '*2', '*3B', '*3C', '*4']
tpmt_calls_fixed = call_star_alleles_fixed(variant_clean, 'TPMT', tpmt_no_function, tpmt_priority)

nudt15_priority = ['*3', '*2', '*9']
nudt15_calls_fixed = call_star_alleles_fixed(variant_clean, 'NUDT15', nudt15_no_function, nudt15_priority)

from collections import Counter
print("TPMT calls, fixed (per-patient counts):", Counter(tuple(a) for a in tpmt_calls_fixed.values()))
print("NUDT15 calls, fixed (per-patient counts):", Counter(tuple(a) for a in nudt15_calls_fixed.values()))


In [ ]:
# spot-check the raw genotype calls behind the TPMT*3A and NUDT15*3 carriers
tpmt_3a_carriers = [s for s, a in tpmt_calls_fixed.items() if '*3A' in a]
nudt15_3_carriers = [s for s, a in nudt15_calls_fixed.items() if '*3' in a]

tpmt_carrier_genotypes = variant_clean[
    (variant_clean['Sample'].isin(tpmt_3a_carriers)) &
    (variant_clean['Gene']=='TPMT') &
    (variant_clean['Pos'].astype(str).isin(['18138997', '18130687']))
][['Sample', 'Pos', 'Ref', 'Alt', 'Genotype']]

print(f"TPMT*3A carriers: {len(tpmt_3a_carriers)}, NUDT15*3 carriers: {len(nudt15_3_carriers)}")
print(tpmt_carrier_genotypes.drop(columns=['Sample']))


In [ ]:
# build a clean per-patient genotype summary
genotype_summary = pd.DataFrame({
    'sample': list(tpmt_calls_fixed.keys()),
    'tpmt_star': [','.join(v) for v in tpmt_calls_fixed.values()],
    'nudt15_star': [','.join(nudt15_calls.get(s, ['*1'])) for s in tpmt_calls_fixed.keys()]
})

genotype_summary['tpmt_carrier'] = genotype_summary['tpmt_star'] != '*1'
genotype_summary['nudt15_carrier'] = genotype_summary['nudt15_star'] != '*1'

print(f"TPMT carriers: {genotype_summary['tpmt_carrier'].sum()} / {len(genotype_summary)}")
print(f"NUDT15 carriers: {genotype_summary['nudt15_carrier'].sum()} / {len(genotype_summary)}")

# join to MRD-merged MAGIC-I manifest via the study ID crosswalk
genotype_with_studyid = genotype_summary.merge(
    id_mapping[['study_id', 'sample']],
    on='sample', how='left'
)
print(f"Matched to study_id: {genotype_with_studyid['study_id'].notna().sum()} of {len(genotype_summary)}")

genotype_mrd = genotype_with_studyid.merge(
    mrd_analysis_df[['sample', 'study_id', 'mrd_status']],
    on='study_id', how='left', suffixes=('', '_mrd')
)
print(f"Matched to MRD status: {genotype_mrd['mrd_status'].notna().sum()} of {len(genotype_summary)}")

table_df = genotype_mrd.dropna(subset=['mrd_status']).copy()

table_df['tpmt_star_full'] = table_df['tpmt_star'].apply(lambda x: '*1/*1' if x == '*1' else f'*1/{x}')
table_df['nudt15_star_full'] = table_df['nudt15_star'].apply(lambda x: '*1/*1' if x == '*1' else f'*1/{x}')

table_df['tpmt_carrier'] = table_df['tpmt_carrier'].map({True: 'Yes', False: 'No'})
table_df['nudt15_carrier'] = table_df['nudt15_carrier'].map({True: 'Yes', False: 'No'})

table_df = table_df.rename(columns={
    'sample': 'Sample',
    'tpmt_star_full': 'TPMT Star Allele',
    'nudt15_star_full': 'NUDT15 Star Allele',
    'tpmt_carrier': 'TPMT Carrier',
    'nudt15_carrier': 'NUDT15 Carrier',
    'mrd_status': 'MRD Status'
})

# save the full per-patient table to CSV rather than displaying patient-level identifiers
full_cols = ['Sample', 'TPMT Star Allele', 'NUDT15 Star Allele', 'TPMT Carrier', 'NUDT15 Carrier', 'MRD Status']
table_df[full_cols].to_csv('genotype_mrd_matched_table.csv', index=False)

# figure shows only de-identified summary counts, not per-patient rows
display_cols = ['TPMT Star Allele', 'NUDT15 Star Allele', 'TPMT Carrier', 'NUDT15 Carrier', 'MRD Status']
summary_counts = table_df[display_cols].value_counts().reset_index(name='n_patients')

fig, ax = plt.subplots(figsize=(10, 0.35 * len(summary_counts) + 1))
ax.axis('off')
tbl = ax.table(cellText=summary_counts.values, colLabels=summary_counts.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)
plt.title('Genotype summary, MRD-matched patients (MAGIC-I, de-identified)', pad=20)
plt.tight_layout()
plt.savefig('genotype_mrd_matched_table.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
overlap = set(genotype_summary['sample']) & set(mrd_analysis_df['sample'])
print(f"Direct overlap (sample-ID scheme): {len(overlap)} of {len(genotype_summary)} genotyped samples")


In [ ]:
id_mapping = pd.read_excel("data/Patient_code_germline_sample_name.xlsx", header=None,
                          names=['study_id', 'disease', 'sample', 'status'])

print(id_mapping.shape)

# check overlap
overlap = set(genotype_summary['sample']) & set(id_mapping['sample'])
print(f"Overlap: {len(overlap)} of {len(genotype_summary)}")


In [ ]:
# step 1: bring study_id into genotype_summary
genotype_with_studyid = genotype_summary.merge(
    id_mapping[['sample', 'study_id']], on='sample', how='left'
)
print(f"Matched to study_id: {genotype_with_studyid['study_id'].notna().sum()} of {len(genotype_summary)}")

# step 2: join to mrd_analysis_df via study_id
genotype_mrd = genotype_with_studyid.merge(
    mrd_analysis_df[['study_id', 'mrd_status', 'TPMT_vst', 'NUDT15_vst']],
    on='study_id', how='left'
)

# build full diplotype (*1/*1 or *1/<allele>) since all carriers are heterozygous
genotype_mrd['tpmt_genotype'] = genotype_mrd['tpmt_star'].apply(lambda x: '*1/*1' if x == '*1' else f'*1/{x}')
genotype_mrd['nudt15_genotype'] = genotype_mrd['nudt15_star'].apply(lambda x: '*1/*1' if x == '*1' else f'*1/{x}')

print(f"Matched to MRD data: {genotype_mrd['mrd_status'].notna().sum()} of {len(genotype_summary)}")


In [ ]:
matched = genotype_mrd.dropna(subset=['mrd_status'])
print(f"Genotype + MRD matched patients: {len(matched)}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5))
colors = {'negative': '#378ADD', 'positive': '#E24B4A'}

genotype_order = sorted(matched['tpmt_star'].unique())  # e.g. ['*1', '*3A']
x_positions = {g: i for i, g in enumerate(genotype_order)}

np.random.seed(42)
for genotype in genotype_order:
    subset = matched[matched['tpmt_star'] == genotype]
    for mrd_stat in ['negative', 'positive']:
        sub2 = subset[subset['mrd_status'] == mrd_stat]
        jit = np.random.uniform(-0.12, 0.12, size=len(sub2))
        ax.scatter(x_positions[genotype] + jit, sub2['TPMT_vst'], color=colors[mrd_stat],
                   s=70, alpha=0.85, edgecolors='black', linewidths=0.8)

ax.set_xticks(list(x_positions.values()))
ax.set_xticklabels([f'{g}\n(n={(matched["tpmt_star"]==g).sum()})' for g in genotype_order])
ax.set_ylabel('TPMT expression (VST-normalized)')
ax.set_xlabel('TPMT genotype')
ax.set_title('TPMT expression by genotype and MRD status (MAGIC-I, n=29)')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['negative'], markersize=10, label='MRD negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['positive'], markersize=10, label='MRD positive'),
]
ax.legend(handles=legend_elements, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('tpmt_genotype_mrd_by_star.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

matched = genotype_mrd.dropna(subset=['mrd_status']).copy()
colors = {'negative': '#378ADD', 'positive': '#E24B4A'}
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, gene_col, expr_col, gene_label in [
    (axes[0], 'tpmt_star', 'TPMT_vst', 'TPMT'),
    (axes[1], 'nudt15_star', 'NUDT15_vst', 'NUDT15')
]:
    genotype_order = sorted(matched[gene_col].unique())
    x_positions = {g: i for i, g in enumerate(genotype_order)}

    for genotype in genotype_order:
        subset = matched[matched[gene_col] == genotype]
        for mrd_stat in ['negative', 'positive']:
            sub2 = subset[subset['mrd_status'] == mrd_stat]
            jit = np.random.uniform(-0.12, 0.12, size=len(sub2))
            ax.scatter(x_positions[genotype] + jit, sub2[expr_col], color=colors[mrd_stat],
                       s=70, alpha=0.85, edgecolors='black', linewidths=0.8)

    ax.set_xticks(list(x_positions.values()))
    ax.set_xticklabels([f'{g}\n(n={(matched[gene_col]==g).sum()})' for g in genotype_order])
    ax.set_ylabel(f'{gene_label} expression (VST-normalized)')
    ax.set_xlabel(f'{gene_label} genotype')
    ax.set_title(f'{gene_label} expression by genotype and MRD status')
    ax.spines[['top', 'right']].set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['negative'], markersize=10, label='MRD negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['positive'], markersize=10, label='MRD positive'),
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.savefig('tpmt_nudt15_genotype_mrd.png', dpi=300)
plt.show()

In [ ]:
# read depth/quality check on carrier calls
carrier_ids = tpmt_3a_carriers + nudt15_3_carriers
carrier_variant_check = variant_clean[
    (variant_clean['Sample'].isin(carrier_ids)) &
    (
        ((variant_clean['Gene']=='TPMT') & (variant_clean['Pos'].astype(str).isin(['18138997','18130687']))) |
        ((variant_clean['Gene']=='NUDT15') & (variant_clean['Pos'].astype(str)=='48045719'))
    )
][['Gene','Pos','Genotype','ReadDepth','AlleleDepth','Quality']]
print(carrier_variant_check)


In [ ]:
#allele frequency comparison 
observed_tpmt_3a_freq = 3/41
observed_nudt15_3_freq = 1/41

published_freqs = {
    'TPMT*3A': {'African American':0.008,'Central/South Asian':0.00422,'East Asian':0.00031,
                'European':0.03384,'Latino':0.04173,'Near Eastern':0.01307,'Sub-Saharan African':0.00162},
    'NUDT15*3': {'Central/South Asian':0.067,'East Asian':0.0605,'European':0.002,'Latino':0.0075}
}

print(f"Observed TPMT*3A frequency: {observed_tpmt_3a_freq:.4f} ({observed_tpmt_3a_freq*100:.2f}%)")
print("Published TPMT*3A frequencies by ancestry:")
for pop, freq in published_freqs['TPMT*3A'].items():
    print(f"  {pop}: {freq*100:.2f}%")

print(f"\nObserved NUDT15*3 frequency: {observed_nudt15_3_freq:.4f} ({observed_nudt15_3_freq*100:.2f}%)")
print("Published NUDT15*3 frequencies by ancestry:")
for pop, freq in published_freqs['NUDT15*3'].items():
    print(f"  {pop}: {freq*100:.2f}%")

In [ ]:
# genotype vs fusion/subtype
genotype_fusion = genotype_with_studyid.merge(
    merged[['study_id', 'Fusion']], on='study_id', how='left'
)
carrier_fusion = genotype_fusion[genotype_fusion['tpmt_carrier'] | genotype_fusion['nudt15_carrier']]
print(carrier_fusion[['tpmt_star','nudt15_star','Fusion']])

print("\nOverall fusion distribution for comparison:")
print(merged['Fusion'].value_counts(dropna=False))


In [ ]:
# Rebuild mrd_analysis_df now that full_df has pathway scores merged in
mrd_analysis_df = full_df[full_df['mrd_status'].notna()].copy()
print(mrd_analysis_df.columns.tolist())

In [ ]:
genotype_pathways = genotype_with_studyid.merge(
    mrd_analysis_df[['study_id'] + pathway_cols], on='study_id', how='left'
)
carrier_pathways = genotype_pathways[genotype_pathways['tpmt_carrier']]
print(carrier_pathways[pathway_cols])

print("\nCohort-wide means for comparison:")
print(mrd_analysis_df[pathway_cols].mean())


In [ ]:
tpmt_uncertain_positions = ['18147910','18133845','18130725','18130762','18143606','18139027',
                             '18139710','18149045','18138969','18147845','18143597','18130694']

uncertain_hits = variant_clean[
    (variant_clean['Gene']=='TPMT') & (variant_clean['Pos'].astype(str).isin(tpmt_uncertain_positions))
][['Pos','Ref','Alt','Genotype']]
print(f"Uncertain-function TPMT variant hits: {len(uncertain_hits)}")
print(uncertain_hits)


1. pathway score heatmap (29 MRD-matched patients x 9 pathways, with genotype/MRD annotations

In [ ]:
labels = [f"Patient {n+1}{' (*3A)' if c else ''}" for n, c in enumerate(heatmap_data['tpmt_carrier'])]
ax.set_yticklabels(labels, fontsize=8)


All 3 TPMT*3A carriers sit predominantly in the pale/light blue range across most pathways. No clear directional pattern in pathway activity among TPMT*3A carriers. 

2. Variant presence/absence heatmap (all 205 positions x 41 patients)

In [ ]:
ax.set_yticklabels([f'Patient {n+1}' for n in range(len(pivot))], fontsize=6)


In [ ]:
# check for near-empty rows (samples with very few variant calls)
variant_counts_per_sample = variant_clean.groupby('Sample').size()
low_count_samples = variant_counts_per_sample[variant_counts_per_sample <= variant_counts_per_sample.quantile(0.05)]
print(f"Samples with unusually low variant counts: {len(low_count_samples)}")
print(variant_counts_per_sample.describe())


3. Pathway-pathway correlation heatmap

In [ ]:
from scipy.stats import spearmanr

corr_mat = np.zeros((len(pathway_cols), len(pathway_cols)))
for i, c1 in enumerate(pathway_cols):
    for j, c2 in enumerate(pathway_cols):
        corr_mat[i,j] = spearmanr(mrd_analysis_df[c1], mrd_analysis_df[c2])[0]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
labels = [c.replace('_score','').replace('_',' ') for c in pathway_cols]
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr_mat[i,j]:.2f}", ha='center', va='center', fontsize=7,
                color='white' if abs(corr_mat[i,j])>0.5 else 'black')
plt.colorbar(im, ax=ax, label="Spearman's rho", shrink=0.8)
ax.set_title('Pathway score correlations (MAGIC-I, n=43)')
plt.tight_layout()
plt.savefig('pathway_correlation_heatmap.png', dpi=300)
plt.show()

Shows how correlated the 9 pathway modules scores are with each other across 43 MAGIC-I patients. 

1. Two tight clusters: 
Cluster A: ribosome + oxidative phosphorylation (rho=0.88)
cluster B: TNF + apoptosis + p53 pathway (rho=0.77-0.87). TNF signalling, apoptosis and p53 are functionally linked in stress/death response biology

2. Glutathione, ferroptosis, purine metabolism, and mTORC1 signalling form a looser, moderately-correlated middle group (rho ~0.4-0.75)

In [ ]:
print(f"Carriers found in vst_df sample-ID scheme: {len(carrier_vst_samples)}")


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(pca_df[~pca_df['tpmt_carrier']]['PC1'], pca_df[~pca_df['tpmt_carrier']]['PC2'],
           color='#378ADD', alpha=0.7, s=50, label='Non-carrier')
ax.scatter(pca_df[pca_df['tpmt_carrier']]['PC1'], pca_df[pca_df['tpmt_carrier']]['PC2'],
           color='#E24B4A', alpha=0.9, s=90, edgecolors='black', label='TPMT*3A carrier')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of MAGIC-I expression, colored by TPMT genotype')
ax.legend()
plt.tight_layout()
plt.savefig('pca_genotype.png', dpi=300)
plt.show()

#### eQTL ####

In [ ]:
from scipy.stats import mannwhitneyu

# merge on study_id instead of sample — same fix you used before
geno_expr = genotype_pathways.merge(
    mrd_analysis_df[['study_id', 'TPMT_vst', 'NUDT15_vst']],
    on='study_id', how='left'
)

print(geno_expr['TPMT_vst'].isna().sum(), "missing TPMT_vst out of", len(geno_expr))

comparisons = [
    ('tpmt_carrier', 'TPMT_vst', "TPMT*3A carrier -> TPMT expression"),
    ('tpmt_carrier', 'NUDT15_vst', "TPMT*3A carrier -> NUDT15 expression"),
    ('nudt15_carrier', 'TPMT_vst', "NUDT15*3 carrier -> TPMT expression"),
    ('nudt15_carrier', 'NUDT15_vst', "NUDT15*3 carrier -> NUDT15 expression"),
]

for carrier_col, expr_col, label in comparisons:
    carriers = geno_expr[geno_expr[carrier_col] == True][expr_col].dropna()
    non_carriers = geno_expr[geno_expr[carrier_col] == False][expr_col].dropna()

    print(f"--- {label} ---")
    print(f"n carriers = {len(carriers)}, n non-carriers = {len(non_carriers)}")
    if len(carriers) >= 2 and len(non_carriers) >= 2:
        stat, p = mannwhitneyu(carriers, non_carriers)
        print(f"carrier median = {carriers.median():.3f}, non-carrier median = {non_carriers.median():.3f}")
        print(f"p = {p:.4f}\n")
    else:
        print("Too few carriers for a valid test — reporting descriptively only.\n")
        print(f"carrier values: {carriers.tolist()}\n")

In [ ]:
print(variant_df.columns.tolist())
print(variant_df['Gene'].unique())


In [ ]:
import requests

def get_grch38_position(rsid):
    url = f"https://rest.ensembl.org/variation/human/{rsid}?content-type=application/json"
    r = requests.get(url)
    data = r.json()
    for mapping in data['mappings']:
        if mapping['assembly_name'] == 'GRCh38':
            return mapping['start'], mapping['seq_region_name']
    return None, None

# TPMT*3A = rs1800460 (c.460G>A) + rs1142345 (c.719A>G)
pos_460, chr_460 = get_grch38_position('rs1800460')
pos_719, chr_719 = get_grch38_position('rs1142345')

# NUDT15*3 = rs116855232 (c.415C>T)
pos_415, chr_415 = get_grch38_position('rs116855232')

print(f"TPMT c.460G>A (rs1800460): chr{chr_460}:{pos_460}")
print(f"TPMT c.719A>G (rs1142345): chr{chr_719}:{pos_719}")
print(f"NUDT15 c.415C>T (rs116855232): chr{chr_415}:{pos_415}")

In [ ]:
carriers_df = genotype_summary[genotype_summary['tpmt_carrier'] | genotype_summary['nudt15_carrier']]
print(carriers_df[['tpmt_star', 'tpmt_zygosity', 'nudt15_star', 'nudt15_zygosity']])


In [ ]:
summary_table = pd.DataFrame({
    'Allele': ['TPMT*3A', 'NUDT15*3'],
    'Frequency': [
        f"{genotype_summary['tpmt_carrier'].sum()} / {len(genotype_summary)} ({genotype_summary['tpmt_carrier'].mean()*100:.1f}%)",
        f"{genotype_summary['nudt15_carrier'].sum()} / {len(genotype_summary)} ({genotype_summary['nudt15_carrier'].mean()*100:.1f}%)",
    ],
    'Zygosity': [
        'All heterozygous (Intermediate Metabolizer)',
        'Heterozygous (Intermediate Metabolizer)',
    ]
})
summary_table

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 2))
ax.axis('off')

star_allele_table_data = [
    ['Allele', 'Frequency', 'Zygosity'],
    ['TPMT*3A', f"{genotype_summary['tpmt_carrier'].sum()} / {len(genotype_summary)} ({genotype_summary['tpmt_carrier'].mean()*100:.1f}%)", 'All heterozygous\n(Intermediate Metabolizer)'],
    ['NUDT15*3', f"{genotype_summary['nudt15_carrier'].sum()} / {len(genotype_summary)} ({genotype_summary['nudt15_carrier'].mean()*100:.1f}%)", 'Heterozygous\n(Intermediate Metabolizer)'],
]

table = ax.table(cellText=star_allele_table_data[1:], colLabels=star_allele_table_data[0], cellLoc='left', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

for i in range(3):
    table.auto_set_column_width(i)

plt.tight_layout()
plt.savefig('star_allele_table.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mannwhitneyu

def run_cis_eqtl_scan(gene, expr_col):
    gene_variants = variants_pass[variants_pass['Gene'] == gene]
    results = []

    for pos in gene_variants['Pos'].unique():
        carriers = gene_variants[gene_variants['Pos'] == pos]['Sample'].unique()

        expr_data = geno_expr[['sample', expr_col]].dropna()
        expr_data['has_variant'] = expr_data['sample'].isin(carriers)

        carrier_vals = expr_data[expr_data['has_variant']][expr_col]
        noncarrier_vals = expr_data[~expr_data['has_variant']][expr_col]

        if len(carrier_vals) < 2 or len(noncarrier_vals) < 2:
            continue  # too few carriers to test

        stat, p = mannwhitneyu(carrier_vals, noncarrier_vals)
        direction = carrier_vals.median() - noncarrier_vals.median()  # + = increases expression

        results.append({'pos': pos, 'pValue': p, 'direction': direction, 'n_carriers': len(carrier_vals)})

    return pd.DataFrame(results)

tpmt_magici_eqtl = run_cis_eqtl_scan('TPMT', 'TPMT_vst')

from statsmodels.stats.multitest import multipletests

reject, pvals_fdr, _, _ = multipletests(tpmt_magici_eqtl['pValue'], method='fdr_bh')
tpmt_magici_eqtl['padj_fdr'] = pvals_fdr
tpmt_magici_eqtl['significant_fdr'] = reject

n_sig = tpmt_magici_eqtl['significant_fdr'].sum()
print(f"{n_sig} of {len(tpmt_magici_eqtl)} positions survive FDR correction (BH, q<0.05)")
print(tpmt_magici_eqtl.sort_values('pValue').head(10))

fig, ax = plt.subplots(figsize=(12, 6))
tpmt_magici_eqtl['neg_log10_p'] = -np.log10(tpmt_magici_eqtl['pValue'])
colors = ['crimson' if x < 0 else 'steelblue' for x in tpmt_magici_eqtl['direction']]
ax.scatter(tpmt_magici_eqtl['pos'], tpmt_magici_eqtl['neg_log10_p'], c=colors, alpha=0.6, s=40)
ax.axvspan(18120000, 18165000, alpha=0.15, color='green', label='Star-allele genotyping window')
ax.set_xlabel('Position on chr6 (GRCh38)')
ax.set_ylabel('-log10(p-value)')
ax.set_title('MAGIC-I cis-eQTL scan: TPMT locus variants vs TPMT expression (n=29)')
ax.legend()
ax.text(0.02, 0.98, 'Blue = increases expression\nRed = decreases expression\n(within-cohort, n=29 matched samples)',
        transform=ax.transAxes, verticalalignment='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig('magic_i_tpmt_cis_eqtl_scan.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import fisher_exact

# TPMT*3A carrier vs MRD status
table = [
    [1, 2],    # carriers: MRD+, MRD-
    [6, 20],   # non-carriers: MRD+, MRD-
]
odds_ratio, p = fisher_exact(table)
print(f"TPMT*3A carrier vs MRD status: OR={odds_ratio:.3f}, p={p:.4f}")

In [ ]:
mrd_analysis_df[['sample', 'mrd_status']].to_csv('magic_i_mrd_status.csv', index=False)
print(mrd_analysis_df['mrd_status'].value_counts())

In [ ]:
thiopurine_genes = ['TPMT', 'NUDT15', 'ITPA', 'HPRT1', 'IMPDH1', 'IMPDH2', 'GMPS', 'ABCC4', 'TYMS']

thiopurine_ids = {g: symbol_to_id[g] for g in thiopurine_genes if g in symbol_to_id}
missing = [g for g in thiopurine_genes if g not in symbol_to_id]
if missing:
    print(f"Not found in symbol_to_id mapping: {missing}")

magic_thiopurine = results_df_magic.loc[
    results_df_magic.index.isin(thiopurine_ids.values())
].copy()
magic_thiopurine['gene_symbol'] = magic_thiopurine.index.map(id_to_symbol)
magic_thiopurine = magic_thiopurine[['gene_symbol', 'log2FoldChange', 'pvalue', 'padj']]
magic_thiopurine = magic_thiopurine.sort_values('padj')

print(magic_thiopurine)
magic_thiopurine.to_csv('thiopurine_pathway_magic_mrd_pvalues.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

table_df = magic_thiopurine.copy()
table_df['log2FoldChange'] = table_df['log2FoldChange'].round(3)
table_df['pvalue'] = table_df['pvalue'].apply(lambda x: f"{x:.3f}")
table_df['padj'] = table_df['padj'].apply(lambda x: f"{x:.3f}")

table_df = table_df.rename(columns={
    'gene_symbol': 'Gene',
    'log2FoldChange': 'log2FC',
    'pvalue': 'p-value',
    'padj': 'Adjusted p-value'
})

fig, ax = plt.subplots(figsize=(8, 0.4 * len(table_df) + 1))
ax.axis('off')
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
plt.title('Thiopurine pathway genes — MRD+ vs MRD- (MAGIC-I)', pad=20)
plt.tight_layout()
plt.savefig('thiopurine_pathway_magic_table.png', dpi=300, bbox_inches='tight')
plt.show()

#### MAGIC-I B-ALL cohort for comparison with TARGET cohort ####

1. B-ALL count matrix + VST 

In [ ]:
qc_exclude_strict = ['A3873043']  # confirmed QC failure

ball_samples_list = [s for s in leukaemia_manifest_narrow[leukaemia_manifest_narrow['lineage'] == 'B-ALL']['sample'] if s not in qc_exclude_strict]
print(f"B-ALL samples (post-QC): n={len(ball_samples_list)}")

counts_ball = counts_leukaemia_narrow[ball_samples_list]
print(counts_ball.shape)

# VST normalization, B-ALL only
metadata_ball_vst = pd.DataFrame(index=ball_samples_list)
dds_ball_vst = DeseqDataSet(
    counts=counts_ball.T,
    metadata=metadata_ball_vst,
    design="~1",  # no covariate needed just for VST
)
dds_ball_vst.deseq2()
dds_ball_vst.vst()
vst_ball = pd.DataFrame(dds_ball_vst.layers['vst_counts'], index=counts_ball.T.index, columns=counts_ball.T.columns)

print(vst_ball.shape)
print(vst_ball.iloc[:5, :5])

2. TPMT/NUDT15 expression histograms + MRD status merge 

In [ ]:
import matplotlib.pyplot as plt

tpmt_ball = vst_ball['ENSG00000137364']
nudt15_ball = vst_ball['ENSG00000136159']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(tpmt_ball, bins=15, color='#4C72B0', edgecolor='white')
axes[0].set_title('TPMT expression (VST) — MAGIC-I B-ALL')
axes[0].set_xlabel('VST')
axes[0].set_ylabel('Number of patients')

axes[1].hist(nudt15_ball, bins=15, color='#DD8452', edgecolor='white')
axes[1].set_title('NUDT15 expression (VST) — MAGIC-I B-ALL')
axes[1].set_xlabel('VST')

plt.tight_layout()
plt.savefig('ball_tpmt_nudt15_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

# merge with MRD status
tpmt_nudt15_ball = pd.DataFrame({'TPMT_vst': tpmt_ball, 'NUDT15_vst': nudt15_ball}).reset_index().rename(columns={'index': 'sample'})
tpmt_nudt15_ball = tpmt_nudt15_ball.merge(mrd_analysis_df[['sample', 'mrd_status']], on='sample', how='left')

print(tpmt_nudt15_ball['mrd_status'].value_counts(dropna=False))
print(tpmt_nudt15_ball.head())

3. Mann-Whitney U test 

In [ ]:
from scipy.stats import mannwhitneyu

matched_ball = tpmt_nudt15_ball.dropna(subset=['mrd_status'])

pos = matched_ball[matched_ball['mrd_status'] == 'positive']
neg = matched_ball[matched_ball['mrd_status'] == 'negative']

tpmt_stat, tpmt_p = mannwhitneyu(pos['TPMT_vst'], neg['TPMT_vst'])
nudt15_stat, nudt15_p = mannwhitneyu(pos['NUDT15_vst'], neg['NUDT15_vst'])

print("=== B-ALL only (n=30) ===")
print(f"TPMT: MRD+ median={pos['TPMT_vst'].median():.3f}, MRD- median={neg['TPMT_vst'].median():.3f}, p={tpmt_p:.4f}")
print(f"NUDT15: MRD+ median={pos['NUDT15_vst'].median():.3f}, MRD- median={neg['NUDT15_vst'].median():.3f}, p={nudt15_p:.4f}")

4. TPMT subtype-adjusted regression (using fusion_status as MAGIC-I equivalent)

In [ ]:
leukaemia_manifest_narrow = leukaemia_manifest_narrow.merge(
    merged[['sample', 'Fusion']], on='sample', how='left'
)
print(leukaemia_manifest_narrow['Fusion'].value_counts(dropna=False))

In [ ]:
fusion_clean = leukaemia_manifest_narrow['Fusion'].str.strip()
fusion_clean = fusion_clean.replace({'No': 'Fusion-negative', 'no fusion': 'Fusion-negative'})
leukaemia_manifest_narrow['fusion_clean'] = fusion_clean
print(leukaemia_manifest_narrow['fusion_clean'].value_counts(dropna=False))

In [ ]:
import statsmodels.formula.api as smf

reg_df_ball = matched_ball.merge(
    leukaemia_manifest_narrow[['sample', 'fusion_clean']], on='sample', how='left'
)
reg_df_ball['fusion_status'] = reg_df_ball['fusion_clean'].apply(
    lambda x: 'fusion_negative' if pd.isna(x) or 'negative' in str(x).lower() else 'fusion_positive'
)

model_tpmt_ball = smf.ols('TPMT_vst ~ mrd_status + fusion_status', data=reg_df_ball).fit()
print(model_tpmt_ball.summary())

5. same regression for NUDT15

In [ ]:
model_nudt15_ball = smf.ols('NUDT15_vst ~ mrd_status + fusion_status', data=reg_df_ball).fit()
print(model_nudt15_ball.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

models = [model_tpmt_ball, model_nudt15_ball]
gene_labels = ['TPMT', 'NUDT15']
predictors = ['mrd_status[T.positive]', 'fusion_status[T.fusion_positive]']
predictor_labels = ['MRD+ status', 'Fusion positive']

rows = []
row_labels = []
for gene, m in zip(gene_labels, models):
    for pred, pred_label in zip(predictors, predictor_labels):
        rows.append({
            'coef': m.params[pred],
            'ci_low': m.conf_int().loc[pred, 0],
            'ci_high': m.conf_int().loc[pred, 1],
            'pval': m.pvalues[pred],
        })
        row_labels.append(f'{gene}: {pred_label}')

y_pos = np.arange(len(rows))
coefs = [r['coef'] for r in rows]
ci_low = [r['ci_low'] for r in rows]
ci_high = [r['ci_high'] for r in rows]
pvals = [r['pval'] for r in rows]

colors = ['#1E2761', '#028090'] * 2  # navy for MRD, teal for fusion, alternating per gene

ax.errorbar(
    coefs, y_pos,
    xerr=[np.array(coefs) - np.array(ci_low), np.array(ci_high) - np.array(coefs)],
    fmt='none', ecolor='#888', elinewidth=2, capsize=4
)
ax.scatter(coefs, y_pos, c=colors, s=100, zorder=3)

ax.axvline(0, color='grey', linestyle='--', linewidth=1)
ax.axhline(1.5, color='#ddd', linewidth=1)  # divider between TPMT and NUDT15 groups

ax.set_yticks(y_pos)
ax.set_yticklabels(row_labels, fontsize=11)
ax.set_xlabel('Coefficient (95% CI)', fontsize=12)
ax.set_title('TPMT/NUDT15 expression: fusion-adjusted regression coefficients\n(MAGIC-I B-ALL only, n=30)', fontsize=12)

for i, (c, p) in enumerate(zip(coefs, pvals)):
    ax.text(c, i - 0.17, f'p={p:.3f}', ha='center', va='top', fontsize=9, color='#444')

ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('ball_tpmt_nudt15_full_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()

6. DESeq2 on B-ALL only 

In [ ]:
mrd_samples_ball = matched_ball['sample'].tolist()
counts_mrd_ball = counts_ball[mrd_samples_ball].T
metadata_ball = matched_ball.set_index('sample').loc[mrd_samples_ball, ['mrd_status']]

print(counts_mrd_ball.shape)
print(metadata_ball['mrd_status'].value_counts())

dds_ball = DeseqDataSet(
    counts=counts_mrd_ball,
    metadata=metadata_ball,
    design="~mrd_status",
    refit_cooks=True
)
dds_ball.deseq2()

stat_res_ball = DeseqStats(dds_ball, contrast=["mrd_status", "positive", "negative"])
stat_res_ball.summary()
results_df_ball = stat_res_ball.results_df

print(results_df_ball.shape)
print(results_df_ball.sort_values('padj').head(10))

In [ ]:
target_sig = pd.read_csv("data/TARGET_DE_results_MRDpos_vs_MRDneg_filtered.csv")
print(target_sig.columns.tolist())
print(target_sig.shape)

In [ ]:
# Map MAGIC-I B-ALL results to gene symbols
ball_sig = results_df_ball.copy()
ball_sig['gene_name'] = ball_sig.index.map(id_to_symbol)
ball_sig = ball_sig.dropna(subset=['gene_name'])
ball_sig = ball_sig[ball_sig['padj'] < 0.05]

print(f"MAGIC-I B-ALL significant genes (padj<0.05): {len(ball_sig)}")

# Find overlap
overlap = set(target_sig['gene_name']) & set(ball_sig['gene_name'])
print(f"\nOverlap between TARGET and MAGIC-I B-ALL significant genes: {len(overlap)}")
print(sorted(overlap))

# Check if TPMT/NUDT15 are in either list (should be in neither, given your null results)
print(f"\nTPMT in TARGET sig: {'TPMT' in target_sig['gene_name'].values}")
print(f"TPMT in MAGIC-I B-ALL sig: {'TPMT' in ball_sig['gene_name'].values}")

In [ ]:
from scipy.stats import hypergeom

# total genes tested in both datasets (use the common background)
total_genes = len(set(ball_sig.index) | set(target_sig['gene_name']))  # rough estimate; better to use actual tested-gene universe
M = 60295  # total genes tested in MAGIC-I (adjust if TARGET's tested-gene count differs)
n = len(target_sig)  # TARGET significant genes
N = len(ball_sig)    # MAGIC-I B-ALL significant genes
k = len(overlap)     # observed overlap

p_value = hypergeom.sf(k - 1, M, n, N)
print(f"Observed overlap: {k} genes")
print(f"Expected by chance: {n * N / M:.2f} genes")
print(f"Hypergeometric p-value: {p_value:.4f}")

The overlap between TARGET and MAGIC-I B-ALL DE genes (7 observed vs 2.45 expected by chance) is significant (p=0.0112). 

TPMT/NUDT15 no replication in MAGIC-I (p=0.924, p=0.848)

Broader MRD-associated transcriptional signature:partial statistically significant replication (7/68 genes overlap TARGET's list, more than expected by chance, p=0.0111).

MRD does drive some consistent, reproducible transcriptional changes across cohorts but specific TPMT-MRD association identified in TARGET is not among them. Suggests TARGET's TPMT finding may be cohort-specific. 

7 overlapping genes: 
ALKAL2, BIRC7, DSC3, GPR176, KRT1, NCKAP5, NPR3

In [ ]:
overlap_genes_symbols = ['ALKAL2', 'BIRC7', 'DSC3', 'GPR176', 'KRT1', 'NCKAP5', 'NPR3']
overlap_ensembl_ids = [symbol_to_id[g] for g in overlap_genes_symbols if g in symbol_to_id]

magic_overlap_stats = results_df_ball.loc[overlap_ensembl_ids, ['log2FoldChange', 'padj']]
magic_overlap_stats.index = magic_overlap_stats.index.map(id_to_symbol)
print(magic_overlap_stats)

7. GSEA

In [ ]:
# Rank genes by Wald statistic (same approach as TARGET and whole-cohort MAGIC-I)
ranked_genes_ball = results_df_ball.dropna(subset=['stat']).sort_values('stat', ascending=False)

# Map to gene symbols
ranking_ball = ranked_genes_ball['stat']
ranking_ball.index = ranked_genes_ball.index.map(lambda x: id_to_symbol.get(x, x))

print(ranking_ball.head())
print(f"Mapped: {sum(ranking_ball.index != ranked_genes_ball.index)} of {len(ranking_ball)}")

In [ ]:
gsea_ball = gp.prerank(
    rnk=ranking_ball,
    gene_sets=["KEGG_2021_Human", "MSigDB_Hallmark_2020"],
    threads=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    outdir="gsea_ball_output",
    seed=42,
    verbose=False
)

gsea_ball_df = gsea_ball.res2d.sort_values("NES", ascending=False)
gsea_ball_sig = gsea_ball_df[gsea_ball_df["FDR q-val"] < 0.25].copy()

print(gsea_ball_sig[['Term', 'NES', 'FDR q-val']].head(20))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

gsea_top = pd.DataFrame({
    "Term": ["Interferon Alpha Response", "Interferon Gamma Response", "Allograft rejection",
             "Th17 cell differentiation", "RIG-I-like receptor signaling", "Inflammatory bowel disease",
             "TNF-alpha Signaling via NF-kB", "Measles", "Th1 and Th2 cell differentiation",
             "Cytosolic DNA-sensing pathway", "Graft-versus-host disease", "Allograft Rejection (Hallmark)",
             "NK cell mediated cytotoxicity", "PD-L1 expression and PD-1 checkpoint", "IL-6/JAK/STAT3 Signaling",
             "Influenza A", "Type I diabetes mellitus", "T cell receptor signaling", "Epstein-Barr virus infection",
             "NOD-like receptor signaling"],
    "NES": [3.168, 3.037, 2.212, 2.162, 2.157, 2.097, 2.058, 2.044, 1.970, 1.950,
            1.948, 1.910, 1.851, 1.818, 1.818, 1.815, 1.806, 1.790, 1.786, 1.744],
    "FDR": [0.000, 0.000, 0.00271, 0.00136, 0.00090, 0.00136, 0.000, 0.00217, 0.00407, 0.00387,
            0.00339, 0.000, 0.00844, 0.00895, 0.00131, 0.00937, 0.00881, 0.00897, 0.00852, 0.01266]
})

gsea_top = gsea_top.sort_values("NES")

fig, ax = plt.subplots(figsize=(9, 8))
scatter = ax.scatter(
    gsea_top["NES"],
    range(len(gsea_top)),
    c=-gsea_top["FDR"].apply(lambda x: max(x, 1e-10)).apply(lambda x: __import__('math').log10(x)),
    cmap="RdYlBu_r",
    s=120,
    edgecolors="grey",
    linewidths=0.5
)
ax.set_yticks(range(len(gsea_top)))
ax.set_yticklabels(gsea_top["Term"], fontsize=9)
ax.set_xlabel("Normalised Enrichment Score (NES)", fontsize=11)
ax.set_title("GSEA — MRD+ vs MRD- (MAGIC-I B-ALL, n=30)", fontsize=11)
ax.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
plt.colorbar(scatter, label="-log10(FDR q-value)")
plt.tight_layout()
plt.savefig("magic_i_ball_gsea_dotplot.png", dpi=300, bbox_inches="tight")
plt.show()

Same immune-dominated hits persist even after restricting to B-ALL. This is real biological signal associated with MRD status, not confound.

TNF-alpha Signalling via NF-kB appears --> same pathway one of TARGET's indepenent findings predicting MRD status. Second line of pathway-level replication evidence. 



8. pathway module scores

In [ ]:
def compute_pathway_score(gene_symbols, vst_data, symbol_to_id_map):
    ensembl_ids = [symbol_to_id_map[g] for g in gene_symbols if g in symbol_to_id_map]
    present = [g for g in ensembl_ids if g in vst_data.columns]
    sub = vst_data[present].T
    z = sub.subtract(sub.mean(axis=1), axis=0).divide(sub.std(axis=1), axis=0)
    return z.mean(axis=0), len(present)

pathway_scores_ball = {}
for label, genes in [
    ('ribosome_score', ribosome_genes),
    ('glutathione_score', glutathione_genes),
    ('ferroptosis_score', ferroptosis_genes),
    ('tnf_score', tnf_genes),
    ('purine_metabolism_score', purine_metabolism_genes if 'purine_metabolism_genes' in dir() else None),
]:
    if genes is not None:
        score, n_found = compute_pathway_score(genes, vst_ball, symbol_to_id)
        pathway_scores_ball[label] = score
        print(f"{label}: {n_found} genes found")

pathway_scores_ball_df = pd.DataFrame(pathway_scores_ball)
pathway_scores_ball_df.index.name = 'sample'
pathway_scores_ball_df = pathway_scores_ball_df.reset_index()
print(pathway_scores_ball_df.head())

In [ ]:
extra_pathways = {
    'Purine metabolism': ('kegg', 'Purine metabolism'),
    'Apoptosis': ('kegg', 'Apoptosis'),
    'Oxidative phosphorylation': ('kegg', 'Oxidative phosphorylation'),
    'mTORC1 signaling': ('hallmark', 'mTORC1 Signaling'),
    'p53 pathway': ('hallmark', 'p53 Pathway'),
}

for label, (source, term) in extra_pathways.items():
    lib = kegg_lib if source == 'kegg' else hallmark_lib
    if term in lib:
        genes = lib[term]
        score, n_found = compute_pathway_score(genes, vst_ball, symbol_to_id)
        pathway_scores_ball_df[label.lower().replace(' ', '_') + '_score'] = pathway_scores_ball_df['sample'].map(score)
        print(f"{label}: {n_found}/{len(genes)} genes found")
    else:
        print(f"{label}: term '{term}' not found in {source} library")

print(pathway_scores_ball_df.columns.tolist())

In [ ]:
pathway_ball_mrd = pathway_scores_ball_df.merge(matched_ball[['sample', 'mrd_status']], on='sample', how='left')
pathway_ball_mrd = pathway_ball_mrd.dropna(subset=['mrd_status'])

pathway_cols = ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score',
                'purine_metabolism_score', 'apoptosis_score', 'oxidative_phosphorylation_score',
                'mtorc1_signaling_score', 'p53_pathway_score']

for col in pathway_cols:
    pos = pathway_ball_mrd[pathway_ball_mrd['mrd_status'] == 'positive'][col]
    neg = pathway_ball_mrd[pathway_ball_mrd['mrd_status'] == 'negative'][col]
    stat, p = mannwhitneyu(pos, neg)
    print(f"{col}: p={p:.4f}, MRD+ median={pos.median():.3f}, MRD- median={neg.median():.3f}")

tnf_score closes to significant here (p=0.190) and matches TARGET's finding directionally. All 9 pathway scores remain null in B-ALL only as well matching whole-cohort result. 

In [ ]:
leukaemia_manifest_narrow = leukaemia_manifest_narrow[
    leukaemia_manifest_narrow['sample'] != 'A3873043'
].copy()

print("MAGIC-I narrow cohort, n =", len(leukaemia_manifest_narrow))
print(leukaemia_manifest_narrow['lineage'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lineage_order = ['B-ALL', 'T-ALL', 'MPAL']
summary = tpmt_nudt15_vst_strict.groupby('lineage')[['TPMT_vst', 'NUDT15_vst']].agg(['mean', 'std', 'count'])
summary = summary.loc[lineage_order]

x = np.arange(len(lineage_order))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
tpmt_means = summary[('TPMT_vst', 'mean')].values
tpmt_stds = summary[('TPMT_vst', 'std')].fillna(0).values
nudt15_means = summary[('NUDT15_vst', 'mean')].values
nudt15_stds = summary[('NUDT15_vst', 'std')].fillna(0).values

ax.bar(x - width/2, tpmt_means, width, yerr=tpmt_stds, capsize=4, label='TPMT', color="#4C72B0")
ax.bar(x + width/2, nudt15_means, width, yerr=nudt15_stds, capsize=4, label='NUDT15', color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels(lineage_order)
ax.set_ylabel("Expression (VST)")
ax.set_title("TPMT and NUDT15 expression by lineage\n(MAGIC-I narrow cohort, n=43)")
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig("magic_i_tpmt_nudt15_by_lineage_corrected.png", dpi=300, bbox_inches="tight")
plt.show()

print(summary)

In [ ]:
from scipy.stats import spearmanr
import pandas as pd

pathway_cols_9 = ['ribosome_score', 'glutathione_score', 'ferroptosis_score', 'tnf_score',
                   'purine_metabolism_score', 'apoptosis_score', 'oxidative_phosphorylation_score',
                   'mtorc1_signaling_score', 'p53_pathway_score']

rows = []
for col in pathway_cols_9:
    rho_tpmt, p_tpmt = spearmanr(full_df[col], full_df['TPMT_vst'])
    rho_nudt15, p_nudt15 = spearmanr(full_df[col], full_df['NUDT15_vst'])
    rows.append({"Pathway": col, "TPMT_rho": rho_tpmt, "TPMT_p": p_tpmt, "NUDT15_rho": rho_nudt15, "NUDT15_p": p_nudt15})

corr_df = pd.DataFrame(rows)
print(corr_df.round(4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pathway_labels = {
    'ribosome_score': 'Ribosome',
    'glutathione_score': 'Glutathione',
    'ferroptosis_score': 'Ferroptosis',
    'tnf_score': 'TNF-alpha',
    'purine_metabolism_score': 'Purine metabolism',
    'apoptosis_score': 'Apoptosis',
    'oxidative_phosphorylation_score': 'Oxidative phosphorylation',
    'mtorc1_signaling_score': 'mTORC1 signaling',
    'p53_pathway_score': 'p53 pathway'
}

corr_df.index = corr_df['Pathway'].map(pathway_labels)

corr_data = corr_df[['TPMT_rho', 'NUDT15_rho']].rename(columns={'TPMT_rho': 'TPMT', 'NUDT15_rho': 'NUDT15'})
pval_data = corr_df[['TPMT_p', 'NUDT15_p']].rename(columns={'TPMT_p': 'TPMT', 'NUDT15_p': 'NUDT15'})

annot = corr_data.round(2).astype(str)
for col in annot.columns:
    for idx in annot.index:
        star = "*" if pval_data.loc[idx, col] < 0.05 else ""
        annot.loc[idx, col] = f"{corr_data.loc[idx, col]:.2f}{star}"

fig, ax = plt.subplots(figsize=(4.5, 6.5))
sns.heatmap(corr_data, annot=annot, fmt="", cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5,
            cbar_kws={"label": "Spearman rho"}, ax=ax)
ax.set_title("TPMT/NUDT15 vs pathway scores\n(MAGIC-I narrow cohort, n=43)")

plt.tight_layout()
plt.savefig("magic_i_tpmt_nudt15_pathway_correlation_9.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
corr_data_h = corr_data.T
annot_h = annot.T

fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(corr_data_h, annot=annot_h, fmt="", cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5,
            cbar_kws={"label": "Spearman rho"}, ax=ax)
ax.set_title("TPMT/NUDT15 vs pathway scores (MAGIC-I narrow cohort, n=43)")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig("magic_i_tpmt_nudt15_pathway_correlation_9_horizontal.png", dpi=300, bbox_inches="tight")
plt.show()

#### Subtype assignment and UMAP visualization (final QC pass, n=39) ####

In [ ]:
ball_samples_list_valid = [s for s in ball_samples_list if s in subtype_scores.index]
tall_samples_list_valid = [s for s in tall_samples_list if s in t_all_scores.index]

print(f"B-ALL: {len(ball_samples_list)} in manifest, {len(ball_samples_list_valid)} with expression data")
print(f"T-ALL: {len(tall_samples_list)} in manifest, {len(tall_samples_list_valid)} with expression data")

ball_assignments = subtype_scores.loc[ball_samples_list_valid, ['assigned_subtype']]
tall_assignments = t_all_scores.loc[tall_samples_list_valid, ['assigned_subtype']]

all_assignments = pd.concat([ball_assignments, tall_assignments])
all_assignments.index.name = 'sample'
all_assignments = all_assignments.reset_index()

print("\nSubtype composition (MAGIC-I, marker-based):")
print(all_assignments['assigned_subtype'].value_counts())

In [ ]:
all_assignments = all_assignments.merge(
    full_df[['sample', 'mrd_status', 'TPMT_vst', 'NUDT15_vst']],
    on='sample', how='left'
)

print("Matched with MRD/expression data:", all_assignments['mrd_status'].notna().sum(), "of", len(all_assignments))

# MRD status by subtype (only subtypes with n>=3 to avoid meaningless small-group tests)
subtype_counts = all_assignments['assigned_subtype'].value_counts()
adequate_subtypes = subtype_counts[subtype_counts >= 3].index

print("\nMRD+ rate by subtype (n>=3 only):")
for sub in adequate_subtypes:
    sub_df = all_assignments[all_assignments['assigned_subtype'] == sub]
    n_pos = (sub_df['mrd_status'] == 'positive').sum()
    n_total = sub_df['mrd_status'].notna().sum()
    print(f"{sub}: {n_pos}/{n_total} MRD+")

# TPMT/NUDT15 by subtype (Kruskal-Wallis across adequate subtypes)
from scipy.stats import kruskal
groups_tpmt = [all_assignments[all_assignments['assigned_subtype']==s]['TPMT_vst'].dropna() for s in adequate_subtypes]
stat, p = kruskal(*groups_tpmt)
print(f"\nTPMT by subtype (Kruskal-Wallis, n>=3 subtypes): p={p:.4f}")

groups_nudt15 = [all_assignments[all_assignments['assigned_subtype']==s]['NUDT15_vst'].dropna() for s in adequate_subtypes]
stat, p = kruskal(*groups_nudt15)
print(f"NUDT15 by subtype (Kruskal-Wallis, n>=3 subtypes): p={p:.4f}")

In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np

# Top-variance genes, same approach as TARGET
top_var_genes = vst_df.var(axis=0).sort_values(ascending=False).head(2000).index
X = vst_df.loc[all_assignments['sample'], top_var_genes]

reducer = umap.UMAP(random_state=42)
embedding = reducer.fit_transform(X)

subtypes_unique = all_assignments['assigned_subtype'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(subtypes_unique)))
color_map = dict(zip(subtypes_unique, colors))

fig, ax = plt.subplots(figsize=(8, 7))
for sub in subtypes_unique:
    mask = all_assignments['assigned_subtype'] == sub
    ax.scatter(embedding[mask.values, 0], embedding[mask.values, 1],
               label=sub, color=color_map[sub], s=60, alpha=0.8)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("UMAP of MAGIC-I samples, colored by marker-based subtype (n=39)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig("magic_i_umap_subtype.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

mrd_colors = {"positive": "#d73027", "negative": "#4575b4"}
for status in ["negative", "positive"]:
    mask = all_assignments['mrd_status'] == status
    ax.scatter(embedding[mask.values, 0], embedding[mask.values, 1],
               label=f"MRD {'+' if status=='positive' else '−'}",
               color=mrd_colors[status], s=60, alpha=0.8)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("UMAP of MAGIC-I samples, colored by MRD status (n=39)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig("magic_i_umap_mrd.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# Left panel: subtype
ax = axes[0]
for sub in subtypes_unique:
    mask = all_assignments['assigned_subtype'] == sub
    ax.scatter(embedding[mask.values, 0], embedding[mask.values, 1],
               label=sub, color=color_map[sub], s=60, alpha=0.8)
ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("Colored by subtype")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

# Right panel: MRD status
ax = axes[1]
mrd_colors = {"positive": "#d73027", "negative": "#4575b4"}
for status in ["negative", "positive"]:
    mask = all_assignments['mrd_status'] == status
    ax.scatter(embedding[mask.values, 0], embedding[mask.values, 1],
               label=f"MRD {'+' if status=='positive' else '−'}",
               color=mrd_colors[status], s=60, alpha=0.8)
ax.set_xlabel("UMAP1")
ax.set_title("Colored by MRD status")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle("UMAP of MAGIC-I samples (n=39)", y=1.02)
plt.tight_layout()
plt.savefig("magic_i_umap_combined.png", dpi=300, bbox_inches="tight")
plt.show()